In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Luego, en la línea 14, cambiá la ruta:
df = pd.read_csv('/content/drive/MyDrive/PF/winemag-data-130k-v2-csv.csv')

In [ ]:
# ============================================================
#  MAPA DE BURBUJAS — REGIONES VITIVINÍCOLAS DE ITALIA Y FRANCIA
#  Para ejecutar en Google Colab
# ============================================================

# ── 0. Instalación (solo en Colab) ──────────────────────────
# !pip install plotly --quiet

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── 1. Cargar datos ─────────────────────────────────────────
df_euro = df[df['country'].isin(['France', 'Italy'])].copy()

# ── 2. Coordenadas de todas las regiones ────────────────────
coordenadas_vino = {
    # ── FRANCIA ──
    'Alsace':                 (48.3182,  7.4416),
    'Beaujolais':             (46.1500,  4.6500),
    'Bordeaux':               (44.8378, -0.5792),
    'Burgundy':               (47.0525,  4.3837),
    'Champagne':              (48.9566,  4.3676),
    'France Other':           (46.6034,  1.8883),   # Centroide de Francia
    'Languedoc-Roussillon':   (43.4000,  3.1000),
    'Loire Valley':           (47.3941,  0.6848),
    'Provence':               (43.9352,  6.0679),
    'Rhône Valley':           (44.9333,  4.8924),
    'Southwest France':       (43.6047,  1.4442),

    # ── ITALIA ──
    'Central Italy':          (42.5000, 12.5000),
    'Italy Other':            (41.9028, 12.4964),   # Roma como referencia central
    'Lombardy':               (45.4642,  9.1900),
    'Northeastern Italy':     (46.0000, 12.0000),
    'Northwestern Italy':     (45.5000,  8.5000),
    'Piedmont':               (45.0500,  7.6667),
    'Sicily & Sardinia':      (37.5990, 14.0154),   # Sicilia
    'Southern Italy':         (40.5000, 16.0000),
    'Tuscany':                (43.7695, 11.2558),
    'Veneto':                 (45.4343, 12.3388),
}

# ── 3. Asignar coordenadas ──────────────────────────────────
df_euro['lat'] = df_euro['province'].map(
    lambda x: coordenadas_vino.get(x, (None, None))[0]
)
df_euro['lon'] = df_euro['province'].map(
    lambda x: coordenadas_vino.get(x, (None, None))[1]
)
df_euro = df_euro.dropna(subset=['lat', 'lon'])

# ── 4. Agrupar datos ─────────────────────────────────────────
euro_data = (
    df_euro
    .groupby(['province', 'lat', 'lon', 'country'])
    .agg(
        promedio_puntos=('points', 'mean'),
        cantidad_vinos=('points', 'count')
    )
    .reset_index()
)

france_data = euro_data[euro_data['country'] == 'France'].copy()
italy_data  = euro_data[euro_data['country'] == 'Italy'].copy()

# ── 5. Paletas y configuración común — Estética Claros ──────
COLORSCALE      = [            # rosa claro → vino → dorado
    [0.0,  '#f7c5c5'],         # bajo  → rosa muy suave
    [0.4,  '#c22d41'],         # media → vino brillante
    [0.75, '#8a052b'],         # alta  → vino oscuro
    [1.0,  '#d4af37'],         # máx.  → dorado
]
BUBBLE_OPACITY  = 0.88
FONT_FAMILY     = "Georgia, serif"
PAPER_BG        = "#fdf5ec"   # crema cálida
PLOT_BG         = "#fdf5ec"
LAND_COLOR      = "#ead9c2"   # tierra beige
OCEAN_COLOR     = "#c8dce8"   # azul acero suave
BORDER_COLOR    = "#9e7e6e"   # marrón cálido
GRID_COLOR      = "#e0c8b0"   # grid beige claro
TEXT_COLOR      = "#2b000b"   # vino oscuro (legible sobre claro)
SUBTEXT_COLOR   = "#7a4510"   # marrón dorado

def normalize_sizes(series, min_px=12, max_px=60):
    """Normaliza los conteos a un rango de tamaños de burbuja."""
    s_min, s_max = series.min(), series.max()
    if s_max == s_min:
        return pd.Series([max_px / 2] * len(series))
    return min_px + (series - s_min) / (s_max - s_min) * (max_px - min_px)

def build_scatter(data, colorscale, label_country):
    """Construye un trace Scattergeo para un país."""
    sizes   = normalize_sizes(data['cantidad_vinos'])
    scores  = data['promedio_puntos']

    hover_text = [
        f"<b>{row.province}</b><br>"
        f"🍷 Vinos: <b>{int(row.cantidad_vinos):,}</b><br>"
        f"⭐ Puntuación media: <b>{row.promedio_puntos:.1f}</b>"
        for _, row in data.iterrows()
    ]

    return go.Scattergeo(
        lat=data['lat'],
        lon=data['lon'],
        mode='markers+text',
        name=label_country,
        marker=dict(
            size=sizes,
            color=scores,
            colorscale=colorscale,
            cmin=87,
            cmax=92,
            opacity=BUBBLE_OPACITY,
            line=dict(width=1.2, color='rgba(43,0,11,0.35)'),
            colorbar=dict(
                title=dict(
                    text="Puntuación<br>media",
                    font=dict(color=TEXT_COLOR, size=12, family=FONT_FAMILY),
                    side='right'
                ),
                tickfont=dict(color=SUBTEXT_COLOR, size=11),
                thickness=14,
                len=0.60,
                bgcolor='rgba(253,245,236,0.8)',
                bordercolor='rgba(158,126,110,0.4)',
                borderwidth=1,
                tickmode='linear',
                tick0=87,
                dtick=1,
                x=1.02
            ),
            showscale=True,
        ),
        text=data['province'],
        textposition='top center',
        textfont=dict(size=10, color=TEXT_COLOR, family=FONT_FAMILY),
        hovertext=hover_text,
        hoverinfo='text',
        hoverlabel=dict(
            bgcolor='#ffffff',
            bordercolor='rgba(43,0,11,0.3)',
            font=dict(color='#2b000b', size=13, family=FONT_FAMILY)
        )
    )

# ── 6. Construir figuras ─────────────────────────────────────

def build_figure(data, country_name, lon_range, lat_range, title_str):
    trace = build_scatter(data, COLORSCALE, country_name)

    fig = go.Figure(trace)

    fig.update_geos(
        scope='europe',
        showland=True,
        landcolor=LAND_COLOR,
        showocean=True,
        oceancolor=OCEAN_COLOR,
        showlakes=True,
        lakecolor=OCEAN_COLOR,
        showrivers=True,
        rivercolor='#1e3a5f',
        showcoastlines=True,
        coastlinecolor=BORDER_COLOR,
        coastlinewidth=0.8,
        showframe=False,
        showcountries=True,
        countrycolor=BORDER_COLOR,
        countrywidth=0.8,
        showsubunits=True,
        subunitcolor='rgba(100,120,150,0.4)',
        subunitwidth=0.5,
        lonaxis=dict(range=lon_range, showgrid=True, gridcolor=GRID_COLOR, gridwidth=0.3),
        lataxis=dict(range=lat_range, showgrid=True, gridcolor=GRID_COLOR, gridwidth=0.3),
        bgcolor=PLOT_BG,
        resolution=50,
    )

    fig.update_layout(
        title=dict(
            text=title_str,
            x=0.5,
            xanchor='center',
            font=dict(size=22, color=TEXT_COLOR, family=FONT_FAMILY),
            pad=dict(b=10)
        ),
        paper_bgcolor=PAPER_BG,
        geo=dict(bgcolor=PLOT_BG),
        font=dict(family=FONT_FAMILY, color=TEXT_COLOR),
        margin=dict(l=10, r=100, t=70, b=30),
        width=900,
        height=620,
        annotations=[
            dict(
                text=(
                    f"<b>{len(data)}</b> regiones · "
                    f"<b>{int(data['cantidad_vinos'].sum()):,}</b> vinos totales · "
                    f"Puntuación media: <b>{data['promedio_puntos'].mean():.1f}</b>"
                ),
                xref='paper', yref='paper',
                x=0.5, y=-0.04,
                showarrow=False,
                font=dict(size=11, color=SUBTEXT_COLOR, family=FONT_FAMILY),
                xanchor='center'
            ),
            dict(
                text="Fuente: Wine Enthusiast Magazine",
                xref='paper', yref='paper',
                x=0.01, y=-0.04,
                showarrow=False,
                font=dict(size=10, color='rgba(122,69,16,0.5)', family=FONT_FAMILY),
                xanchor='left'
            )
        ]
    )

    # Leyenda de tamaño (burbujas de referencia)
    for qty, label in [(500, "500"), (2000, "2 000"), (5000, "5 000+")]:
        size_ref = normalize_sizes(
            pd.Series([qty, data['cantidad_vinos'].min(), data['cantidad_vinos'].max()])
        ).iloc[0]
        fig.add_trace(go.Scattergeo(
            lat=[None], lon=[None],
            mode='markers',
            marker=dict(
                size=size_ref,
                color='rgba(200,200,200,0.25)',
                line=dict(width=1.5, color='rgba(255,255,255,0.4)')
            ),
            name=f"{label} vinos",
            showlegend=True
        ))

    fig.update_layout(
        legend=dict(
            title=dict(text="Cantidad de vinos", font=dict(color=SUBTEXT_COLOR, size=11)),
            x=0.01,
            y=0.99,
            xanchor='left',
            yanchor='top',
            bgcolor='rgba(13,17,23,0.7)',
            bordercolor='rgba(255,255,255,0.1)',
            borderwidth=1,
            font=dict(color=TEXT_COLOR, size=11, family=FONT_FAMILY),
            itemsizing='constant'
        )
    )

    return fig


# ── Francia ─────────────────────────────────────────────────
fig_france = build_figure(
    data        = france_data,
    country_name= 'Francia',
    lon_range   = [-5.5, 10.0],
    lat_range   = [41.5, 51.5],
    title_str   = (
        "🍷 <b>Regiones Vitivinícolas de Francia</b><br>"
        "<sup style='color:#a0aec0'>Volumen (tamaño de burbuja) · Puntuación media (color)</sup>"
    )
)

# ── Italia ───────────────────────────────────────────────────
fig_italy = build_figure(
    data        = italy_data,
    country_name= 'Italia',
    lon_range   = [6.5, 18.5],
    lat_range   = [36.5, 47.5],
    title_str   = (
        "🍷 <b>Regiones Vitivinícolas de Italia</b><br>"
        "<sup style='color:#a0aec0'>Volumen (tamaño de burbuja) · Puntuación media (color)</sup>"
    )
)

# ── 7. Mostrar ───────────────────────────────────────────────
fig_france.show()
fig_italy.show()


In [ ]:
# ============================================================
#  MAPA COROPLÉTICO DE ESTADOS UNIDOS — CALIDAD VITIVINÍCOLA
#  Estética Wine — Para ejecutar en Google Colab
# ============================================================

import pandas as pd
import plotly.graph_objects as go

# ── 1. Cargar y filtrar ──────────────────────────────────────
df_us = df[df['country'] == 'US'].copy()

# ── 2. Mapa de códigos de estado ────────────────────────────
state_codes = {
    'California': 'CA', 'Washington': 'WA', 'Oregon': 'OR',
    'New York': 'NY', 'Virginia': 'VA', 'Michigan': 'MI',
    'Idaho': 'ID', 'Texas': 'TX', 'Pennsylvania': 'PA',
    'New Jersey': 'NJ', 'Missouri': 'MO', 'Colorado': 'CO',
    'New Mexico': 'NM', 'Massachusetts': 'MA', 'Ohio': 'OH',
    'Illinois': 'IL', 'Nevada': 'NV', 'Maryland': 'MD',
    'North Carolina': 'NC', 'Arizona': 'AZ'
}
df_us['state_code'] = df_us['province'].map(state_codes)
df_us = df_us.dropna(subset=['state_code'])

# ── 3. Agrupar datos ─────────────────────────────────────────
us_data = (
    df_us
    .groupby('state_code')
    .agg(
        promedio_puntos=('points', 'mean'),
        cantidad_vinos=('points', 'count')
    )
    .reset_index()
)

# ── 4. Paleta wine ───────────────────────────────────────────
WINE_SCALE = [
    [0.0,  '#f7c5c5'],   # bajo  → rosa suave
    [0.4,  '#c22d41'],   # medio → vino brillante
    [0.75, '#8a052b'],   # alto  → vino oscuro
    [1.0,  '#d4af37'],   # máx.  → dorado
]
PAPER_BG   = "#fdf5ec"   # crema cálida
FONT_FAM   = "Georgia, serif"
TEXT_COLOR = "#2b000b"   # vino oscuro
GOLD       = "#7a4510"   # marrón dorado

# ── 5. Hover personalizado ───────────────────────────────────
us_data['hover'] = (
    "<b>" + us_data['state_code'] + "</b><br>"
    "🍷 Vinos: <b>" + us_data['cantidad_vinos'].apply(lambda x: f"{x:,}") + "</b><br>"
    "⭐ Puntuación media: <b>" + us_data['promedio_puntos'].apply(lambda x: f"{x:.1f}") + "</b>"
)

# ── 6. Construir figura ──────────────────────────────────────
fig_us = go.Figure(go.Choropleth(
    locations=us_data['state_code'],
    locationmode="USA-states",
    z=us_data['promedio_puntos'],
    colorscale=WINE_SCALE,
    zmin=87,
    zmax=92,
    hovertext=us_data['hover'],
    hoverinfo='text',
    hoverlabel=dict(
        bgcolor='#fdf5ec',
        bordercolor='rgba(158,126,110,0.6)',
        font=dict(color=TEXT_COLOR, size=12, family=FONT_FAM)
    ),
    marker_line_color='rgba(158,126,110,0.5)',
    marker_line_width=1.0,
    colorbar=dict(
        title=dict(
            text="Puntuación<br>media",
            font=dict(color=TEXT_COLOR, size=12, family=FONT_FAM),
            side='right'
        ),
        tickfont=dict(color=GOLD, size=11),
        thickness=14,
        len=0.60,
        bgcolor='rgba(253,245,236,0.85)',
        bordercolor='rgba(158,126,110,0.4)',
        borderwidth=1,
        tickmode='linear',
        tick0=87,
        dtick=1,
        x=1.02
    )
))

fig_us.update_layout(
    title=dict(
        text=(
            "🍷 <b>Calidad Vitivinícola por Estado — EE. UU.</b><br>"
            "<sup style='color:#d4af37'>Puntuación media · "
            f"{int(us_data['cantidad_vinos'].sum()):,} reseñas totales</sup>"
        ),
        x=0.5,
        xanchor='center',
        font=dict(size=21, color=TEXT_COLOR, family=FONT_FAM),
    ),
    geo=dict(
        scope='usa',
        bgcolor=PAPER_BG,
        landcolor='#ead9c2',
        lakecolor='#c8dce8',
        showlakes=True,
        showcoastlines=True,
        coastlinecolor='rgba(158,126,110,0.5)',
        coastlinewidth=0.8,
        showframe=False,
        showcountries=False,
        subunitcolor='rgba(158,126,110,0.4)',
        subunitwidth=0.8,
    ),
    paper_bgcolor=PAPER_BG,
    font=dict(family=FONT_FAM, color=TEXT_COLOR),
    margin=dict(l=10, r=110, t=80, b=30),
    width=900,
    height=580,
    annotations=[
        dict(
            text="Fuente: Wine Enthusiast Magazine",
            xref='paper', yref='paper',
            x=0.01, y=-0.03,
            showarrow=False,
            font=dict(size=10, color='rgba(212,175,55,0.5)', family=FONT_FAM),
            xanchor='left'
        )
    ]
)

fig_us.show()


In [ ]:
import pandas as pd
import plotly.express as px
import html

# 1. Carga y preparación del dataset
df_modelo = df

# Crear price_tier si no existe en el CSV crudo
if 'price_tier' not in df_modelo.columns and 'price' in df_modelo.columns:
    def get_tier(price):
        if pd.isna(price):
            return "Sin precio informado"
        elif price < 20:
            return "Económico"
        elif price <= 50:
            return "Gama Media"
        else:
            return "Premium"
    df_modelo['price_tier'] = df_modelo['price'].apply(get_tier)
else:
    df_modelo['price_tier'] = df_modelo['price_tier'].replace({
        "1. Económico (<$20)": "Económico",
        "2. Gama Media ($20-$50)": "Gama Media",
        "3. Premium (>$50)": "Premium",
        "Sin precio informado": "Sin precio informado"
    })

orden_precios = ["Económico", "Gama Media", "Premium", "Sin precio informado"]
orden_precios = [p for p in orden_precios if p in df_modelo['price_tier'].unique()]

# Diccionario fuerte de traducción para ejes y leyendas
labels_esp = {
    "price_tier": "Segmento Comercial",
    "points": "Puntaje de Calidad",
    "price": "Precio Promedio (USD)",
    "vintage_year": "Año de Cosecha",
    "country_esp": "País",
    "province": "Región",
    "Cantidad": "Cantidad de Vinos",
    "Sabor": "Perfil Aromático"
}

traduccion_paises = {
    "US": "Estados Unidos", "France": "Francia", "Italy": "Italia",
    "Spain": "España", "Portugal": "Portugal", "Chile": "Chile",
    "Argentina": "Argentina", "Austria": "Austria", "Australia": "Australia",
    "Germany": "Alemania", "New Zealand": "Nueva Zelanda", "South Africa": "Sudáfrica", "Greece": "Grecia"
}
df_modelo['country_esp'] = df_modelo['country'].replace(traduccion_paises)

# --- SOLUCIÓN DEL ERROR: Definir dummies_sensorial ---
# Buscaremos términos clave comunes en la descripción de los vinos
sabores_buscar = ['oak', 'tannin', 'cherry', 'plum', 'berry', 'vanilla', 'spice', 'fruit', 'acid', 'dry']
df_desc = df_modelo['description'].fillna('').str.lower()
dummies_sensorial = pd.DataFrame()
for sabor in sabores_buscar:
    # Traducimos la columna al español para que el gráfico sea consistente
    sabor_es = {
        'oak': 'Roble', 'tannin': 'Taninos', 'cherry': 'Cereza', 'plum': 'Ciruela',
        'berry': 'Baya', 'vanilla': 'Vainilla', 'spice': 'Especias', 'fruit': 'Fruta',
        'acid': 'Acidez', 'dry': 'Seco'
    }.get(sabor, sabor.capitalize())
    dummies_sensorial[sabor_es] = df_desc.str.contains(sabor).astype(int)

# --- GRÁFICO 1: Boxplot ---
fig1 = px.box(df_modelo, x="price_tier", y="points", color="price_tier",
              category_orders={"price_tier": orden_precios},
              labels=labels_esp,
              color_discrete_sequence=['#ff9999', '#c22d41', '#5c001e', '#b31237'])
fig1.update_traces(hovertemplate='Segmento: %{x}<br>Puntaje: %{y}<extra></extra>')
fig1.update_layout(showlegend=False, template="plotly_white", margin=dict(l=50, r=20, t=20, b=80), font=dict(size=12))
fig1.update_xaxes(title_text=None)

# --- GRÁFICO 2: Barras Horizontales ---
top_sabores = dummies_sensorial.sum().sort_values(ascending=False).head(10)
df_sabores = pd.DataFrame({'Sabor': top_sabores.index, 'Cantidad': top_sabores.values})
fig2 = px.bar(df_sabores, x='Cantidad', y='Sabor', orientation='h',
              labels=labels_esp,
              color='Cantidad', color_continuous_scale=['#ff9999', '#c22d41', '#5c001e'])
fig2.update_layout(yaxis={'categoryorder':'total ascending'}, template="plotly_white",
                   margin=dict(l=100, r=20, t=20, b=60), font=dict(size=12), coloraxis_showscale=False)

# --- GRÁFICO 3: Scatter Plot ---
df_scatter = df_modelo[df_modelo['price'] <= 150]
fig3 = px.scatter(df_scatter, x="price", y="points", color="price_tier",
                  category_orders={"price_tier": orden_precios},
                  labels=labels_esp,
                  color_discrete_sequence=['#ff9999', '#c22d41', '#5c001e', '#b31237'])
fig3.update_traces(hovertemplate='Precio: $%{x}<br>Puntaje: %{y}<extra></extra>')
fig3.update_layout(
    template="plotly_white",
    margin=dict(l=50, r=20, t=20, b=120),
    font=dict(size=12),
    legend=dict(title="Segmento de Mercado:", orientation="h", yanchor="top", y=-0.25, xanchor="center", x=0.5)
)

g1_html = fig1.to_html(full_html=True, include_plotlyjs='cdn', default_height="100%", default_width="100%")
g2_html = fig2.to_html(full_html=True, include_plotlyjs='cdn', default_height="100%", default_width="100%")
g3_html = fig3.to_html(full_html=True, include_plotlyjs='cdn', default_height="100%", default_width="100%")

In [ ]:
# @title Presentación Interactiva Final: Data Winery
# ──────────────────────────────────────────────────────────────
# INSTRUCCIONES:
#   Correr PRIMERO las celdas con:
#     · fig1, fig2, fig3  (gráficos analíticos)
#     · fig_france, fig_italy  (wine_bubble_map.py)
#     · fig_us  (us_choropleth.py)
#   Luego correr ESTA celda.
# ──────────────────────────────────────────────────────────────
import uuid
import base64
import html
from IPython.display import HTML, display

def plot_to_b64(fig):
    html_bytes = fig.to_html(
        full_html=True, include_plotlyjs='cdn',
        default_height="100%", default_width="100%"
    ).encode('utf-8')
    return "data:text/html;base64," + base64.b64encode(html_bytes).decode('utf-8')

# Gráficos analíticos
# Boxplot por srcdoc (html escapado para evitar límite de peso en colab)
src1 = html.escape(fig1.to_html(full_html=True, include_plotlyjs='cdn', default_height="100%", default_width="100%"))
src2 = plot_to_b64(fig2)   # Barras descriptores sensoriales
src3 = plot_to_b64(fig3)   # Scatter precio vs puntaje
# Mapas vitivinícolas
src4 = plot_to_b64(fig_france)  # Mapa burbuja Francia
src5 = plot_to_b64(fig_italy)   # Mapa burbuja Italia
src6 = plot_to_b64(fig_us)      # Coroplético EE. UU.

_html = f'''
<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <title>Data Winery - Pipeline ETL</title>
</head>
<body>
<div class="gx" id="__ID__" tabindex="0">
<style>
.gx {{
    --vino: #5c001e; --vino-oscuro: #2b000b; --dorado: #d4af37;
    --papel: #fcf9f2; --texto: #333; --rosa: #ff9999;
    font-family: 'Segoe UI', system-ui, sans-serif;
    background: var(--papel); color: var(--texto);
    border-radius: 8px; padding: 0; display: flex; flex-direction: column;
    box-sizing: border-box; outline: none;
    width: 100%; max-width: 100%; height: 93vh;
    overflow-y: auto; overflow-x: hidden; position: relative;
}}
.gx * {{ box-sizing: border-box; }}
.gx.fs, .gx:fullscreen {{ width: 100vw; height: 100vh; max-width: none; border-radius: 0; overflow: auto; }}
.gx::-webkit-scrollbar {{ width: 10px; }}
.gx::-webkit-scrollbar-track {{ background: var(--papel); }}
.gx::-webkit-scrollbar-thumb {{ background: var(--vino); border-radius: 5px; }}

/* ── Botón pantalla completa ── */
.gx .sticky-wrapper {{ position: sticky; top: 15px; width: 100%; height: 0; z-index: 100000; display: flex; justify-content: flex-end; padding-right: 15px; pointer-events: none; }}
.gx .btn-fs-mini {{ background: rgba(212,175,55,0.25); color: var(--dorado); border: 1px solid var(--dorado); border-radius: 6px; font-weight: bold; cursor: pointer; transition: all 0.3s ease; pointer-events: auto; width: 42px; height: 42px; display: flex; align-items: center; justify-content: flex-start; padding: 0; box-shadow: 0 4px 10px rgba(0,0,0,0.4); backdrop-filter: blur(3px); overflow: hidden; }}
.gx .fs-icon {{ min-width: 40px; display: flex; align-items: center; justify-content: center; font-size: 1.4em; }}
.gx .fs-text {{ white-space: nowrap; font-size: 0.95em; opacity: 0; transition: opacity 0.3s ease; margin-left: 5px; }}
.gx .btn-fs-mini:hover {{ width: 200px; background: var(--dorado); color: var(--vino-oscuro); }}
.gx .btn-fs-mini:hover .fs-text {{ opacity: 1; }}

/* ── Pantalla de Intro ── */
.gx #__ID__-intro {{ position: absolute; top: 0; left: 0; width: 100%; height: 100%; background-color: var(--vino-oscuro); z-index: 9999; display: flex; flex-direction: column; align-items: center; justify-content: center; transition: opacity 1.5s ease; overflow: hidden; text-align: center; padding: 20px; }}
.gx .intro-bg {{ position: absolute; top: 0; left: 0; width: 100%; height: 100%; z-index: 0; background: #000; }}
.gx .intro-bg img, .gx .carousel-bg img {{ position: absolute; top: 0; left: 0; width: 100%; height: 100%; object-fit: cover; opacity: 0; transition: opacity 1.2s ease-in-out; }}
.gx .intro-bg img.active, .gx .carousel-bg img.active {{ opacity: 1; }}
.gx .intro-overlay, .gx .hero-overlay {{ position: absolute; top: 0; left: 0; width: 100%; height: 100%; background: linear-gradient(rgba(43,0,11,0.85), rgba(92,0,30,0.85)); z-index: 1; }}
.gx .btn-comenzar {{ background-color: var(--dorado); color: var(--vino-oscuro); border: none; position: relative; z-index: 5; padding: 18px 55px; font-size: 1.6em; font-weight: bold; border-radius: 40px; cursor: pointer; box-shadow: 0 10px 25px rgba(0,0,0,0.6); transition: all 0.3s ease; letter-spacing: 2px; text-transform: uppercase; font-family: 'Segoe UI', sans-serif; margin-bottom: 20px; }}
.gx .btn-comenzar:hover {{ background-color: #fff; transform: scale(1.06); }}
.gx #__ID__-introContent {{ display: none; flex-direction: column; align-items: center; justify-content: center; width: 100%; height: 100%; z-index: 4; position: relative; }}
.gx #__ID__-memberStack {{ display: flex; flex-direction: column; align-items: center; gap: 18px; transition: transform 0.6s cubic-bezier(.55,0,.85,.35), opacity 0.5s ease; transform-origin: center center; position: absolute; }}
.gx #__ID__-memberStack.crushed {{ transform: scaleY(0.02); opacity: 0; }}
.gx .member {{ font-family: 'Georgia', serif; font-size: 2.8em; color: #f1ece1; text-shadow: 0 4px 10px rgba(0,0,0,0.5); opacity: 0; transform: translateY(20px); transition: opacity 0.5s ease, transform 0.5s cubic-bezier(.2,.8,.2,1); }}
.gx .member .ln {{ color: var(--dorado); font-weight: bold; }}
.gx #__ID__-trubo-presenta {{ position: absolute; font-family: 'Georgia', serif; font-size: 4.5em; color: var(--dorado); font-style: italic; opacity: 0; transition: opacity 1s ease; text-shadow: 0 10px 30px rgba(0,0,0,0.9); display: none; }}
.gx #__ID__-introTitle {{ position: absolute; text-align: center; width: 90%; opacity: 0; transform: scale(0.9); transition: all 1.5s cubic-bezier(.2,0.8,.2,1); display: none; }}
.gx #__ID__-introTitle.show {{ opacity: 1; transform: scale(1); }}
.gx #__ID__-introTitle h1 {{ font-family: 'Georgia', serif; font-size: 5.5em; color: var(--dorado); margin: 0; text-shadow: 0 15px 40px rgba(0,0,0,0.9); letter-spacing: 4px; line-height: 1.1; text-transform: uppercase; }}
.gx #__ID__-introTitle .sub {{ font-size: 1.8em; color: #fcf9f2; margin-top: 15px; font-style: italic; opacity: 0.9; letter-spacing: 1px; }}

/* ── Contenido principal ── */
.gx #__ID__-main {{ display: none; opacity: 0; transition: opacity 1s ease; width: 100%; height: 100%; }}
.gx .hero {{ position: relative; height: 280px; width: 100%; display: flex; align-items: center; justify-content: center; overflow: hidden; flex-shrink: 0; }}
.gx .carousel-bg {{ position: absolute; top: 0; left: 0; width: 100%; height: 100%; z-index: 0; background: #000; }}
.gx .hero-content {{ position: relative; z-index: 2; text-align: center; color: white; width: 100%; padding: 0 20px; }}
.gx .hero-content h1 {{ font-size: 3.2em; color: var(--dorado); margin: 0 0 10px 0; letter-spacing: 2px; text-transform: uppercase; font-family: 'Georgia', serif; }}
.gx .hero-content p {{ font-size: 1.2em; font-style: italic; margin: 0; opacity: 0.9; }}

/* ── Tabs ── */
.gx .tabs {{ display: flex; width: 100%; background: var(--vino-oscuro); position: sticky; top: 0; z-index: 10; box-shadow: 0 4px 6px rgba(0,0,0,0.2); }}
.gx .tabs button {{ flex: 1; background: none; border: none; color: white; padding: 16px; font-size: 1.05em; cursor: pointer; transition: 0.3s; border-bottom: 4px solid transparent; text-transform: uppercase; letter-spacing: 1px; }}
.gx .tabs button:hover {{ background: rgba(255,255,255,0.1); }}
.gx .tabs button.on {{ border-bottom-color: var(--dorado); color: var(--dorado); font-weight: bold; background: rgba(0,0,0,0.3); }}
.gx .tab-content {{ display: none; padding: 30px 3% 50px 3%; animation: gxin 0.4s; width: 100%; }}
.gx .tab-content.active {{ display: block; }}
.gx h2.seccion {{ color: var(--vino); border-bottom: 2px solid var(--dorado); padding-bottom: 8px; margin-top: 0; font-size: 1.8em; font-family: 'Georgia', serif; }}

/* ── Galería ── */
.gx .graph-gallery {{ position: relative; width: 100%; display: flex; flex-direction: column; align-items: center; margin-top: 15px; padding: 0 20px; }}
.gx .gallery-inner {{ position: relative; width: 100%; max-width: 960px; display: flex; align-items: center; justify-content: center; }}
.gx .gallery-btn {{ position: absolute; top: 45%; transform: translateY(-50%); background: rgba(43,0,11,0.9); color: var(--dorado); border: 2px solid var(--dorado); width: 45px; height: 45px; border-radius: 50%; font-size: 1.4em; display: flex; align-items: center; justify-content: center; cursor: pointer; z-index: 20; transition: 0.3s; box-shadow: 0 4px 10px rgba(0,0,0,0.3); }}
.gx .gallery-btn:hover {{ background: var(--dorado); color: var(--vino-oscuro); transform: translateY(-50%) scale(1.1); }}
.gx .gallery-btn.prev {{ left: -25px; }}
.gx .gallery-btn.next {{ right: -25px; }}
.gx .graph-slide {{ display: none; width: 100%; flex-direction: column; align-items: center; animation: gxin 0.5s; }}
.gx .graph-slide.active {{ display: flex; }}

/* ── Tarjeta de slide (con código) ── */
.gx .slide-card-container {{
    background: white; padding: 28px 34px;
    border-radius: 8px; box-shadow: 0 6px 15px rgba(0,0,0,0.08);
    width: 100%; min-height: 420px;
    display: flex; flex-direction: column; justify-content: flex-start;
    border-left: 6px solid var(--dorado);
    margin-bottom: 15px; overflow-y: auto; max-height: 490px;
}}
.gx .slide-card-container h3 {{ color: var(--vino); margin-top: 0; margin-bottom: 8px; font-size: 1.65em; font-family: 'Georgia', serif; }}
.gx .slide-card-container p {{ font-size: 1.02em; color: #444; line-height: 1.65; margin-bottom: 6px; }}
.gx .slide-card-container code {{ background: rgba(92,0,30,0.08); color: var(--vino); padding: 1px 5px; border-radius: 3px; font-family: 'Consolas', monospace; font-size: 0.9em; }}
.gx .slide-card-container strong {{ color: #2b000b; }}
.gx .slide-card-container em {{ color: #5c001e; font-style: italic; }}

/* ── Bloques de código — Estética Wine ── */
.gx .code-label {{
    display: flex; align-items: center; gap: 7px;
    font-size: 0.70em; font-weight: 700; letter-spacing: 1.3px;
    text-transform: uppercase; color: var(--vino);
    margin-top: 16px; margin-bottom: 4px;
    font-family: 'Segoe UI', sans-serif;
}}
.gx .code-label::before {{
    content: ''; display: inline-block;
    width: 7px; height: 7px; border-radius: 50%;
    background: var(--dorado); flex-shrink: 0;
}}
.gx .code-block {{
    background: #1a0008;
    border: 1px solid rgba(212,175,55,0.3);
    border-left: 3px solid var(--dorado);
    border-radius: 6px;
    padding: 13px 16px;
    font-family: 'Consolas', 'Courier New', monospace;
    font-size: 0.82em;
    color: #f1ece1;
    overflow-x: auto;
    white-space: pre;
    line-height: 1.75;
    box-shadow: inset 0 1px 8px rgba(0,0,0,0.5);
}}
.gx .code-block .kw  {{ color: #ff9999; }}                  /* keywords GREL — rosa */
.gx .code-block .fn  {{ color: #d4af37; }}                  /* funciones  — dorado */
.gx .code-block .str {{ color: #fce8b2; }}                  /* strings    — crema cálida */
.gx .code-block .cm  {{ color: rgba(212,175,55,0.55); font-style: italic; }}  /* comentarios */
.gx .code-block .num {{ color: #e8c97a; }}                  /* números    — ámbar */
.gx .code-block .op  {{ color: #ff9999; }}                  /* operadores — rosa */

/* Grid de 2 columnas para bloques de código */
.gx .code-grid {{ display: grid; grid-template-columns: 1fr 1fr; gap: 14px; width: 100%; }}
.gx .code-grid-col {{ display: flex; flex-direction: column; }}

/* ── Gráficos ── */
.gx .grafico-box.grande {{ background: white; padding: 10px; border-radius: 8px; box-shadow: 0 6px 15px rgba(0,0,0,0.1); width: 100%; height: 530px; overflow: hidden; margin-bottom: 15px; display: flex; justify-content: center; align-items: center; }}
.gx .grafico-box.grande iframe {{ width: 100%; height: 100%; border: none; display: block; }}
.gx .graph-desc {{ background: #fff; padding: 18px; border-radius: 8px; border-left: 5px solid var(--dorado); width: 100%; box-shadow: 0 4px 10px rgba(0,0,0,0.05); }}
.gx .graph-desc h3 {{ color: var(--vino); margin-top: 0; font-size: 1.25em; }}
.gx .graph-desc p {{ margin: 0; font-size: 1.02em; color: #555; line-height: 1.5; }}

/* ── Paginación ── */
.gx .pagination-dots {{ display: flex; gap: 8px; align-items: center; justify-content: center; margin-top: 15px; }}
.gx .dot {{ width: 10px; height: 10px; border-radius: 50%; background-color: #d1c2c7; transition: all 0.3s ease; }}
.gx .dot.active {{ width: 28px; border-radius: 6px; background-color: var(--vino); }}

@keyframes gxin {{ from {{ opacity: 0; transform: translateY(15px); }} to {{ opacity: 1; transform: none; }} }}
</style>

<!-- BOTÓN FLOTANTE -->
<div class="sticky-wrapper">
    <button class="btn-fs-mini" id="__ID__-fs">
        <span class="fs-icon"><svg width="20" height="20" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2.5"><path d="M8 3H5a2 2 0 0 0-2 2v3m18 0V5a2 2 0 0 0-2-2h-3m0 18h3a2 2 0 0 0 2-2v-3M3 16v3a2 2 0 0 0 2 2h3"></path></svg></span>
        <span class="fs-text" id="__ID__-fs-text">Maximizar pantalla</span>
    </button>
</div>

<!-- INTRO -->
<div id="__ID__-intro">
    <div class="intro-bg" id="__ID__-intro-carousel"></div>
    <div class="intro-overlay"></div>
    <div id="__ID__-btnBox" style="z-index:5;">
        <button class="btn-comenzar" onclick="iniciarPresentacion__ID__()">Comenzar</button>
    </div>
    <div id="__ID__-introContent">
        <div id="__ID__-memberStack">
            <div class="member"><span class="ln">CAMPOS</span>, Fiorella</div>
            <div class="member"><span class="ln">MARECO</span>, Carlos</div>
            <div class="member"><span class="ln">SALVATORE</span>, Santiago</div>
            <div class="member"><span class="ln">SITTNER</span>, Germán</div>
        </div>
        <div id="__ID__-trubo-presenta">Grupo Turbo presenta...</div>
        <div id="__ID__-introTitle">
            <h1>ETL &amp; ANÁLISIS CON<br><span style="color:#fff;">OpenRefine</span></h1>
            <div class="sub">Pipeline de Transformación y Modelado sobre Datos Masivos</div>
        </div>
    </div>
</div>

<!-- CONTENIDO PRINCIPAL -->
<div id="__ID__-main">
    <div class="hero">
        <div class="carousel-bg" id="__ID__-main-carousel"></div>
        <div class="hero-overlay"></div>
        <div class="hero-content">
            <h1>ETL &amp; OpenRefine</h1>
            <p>Procesamiento, Limpieza y Estandarización de Datos</p>
        </div>
    </div>

    <div class="tabs" id="__ID__-tabs">
        <button class="on" onclick="openTab__ID__(0)">1. La Tecnología</button>
        <button onclick="openTab__ID__(1)">2. El Dataset</button>
        <button onclick="openTab__ID__(2)">3. Análisis Visual</button>
    </div>

    <!-- ══════════════════════════════════════════════
         PESTAÑA 1 — La Tecnología
    ══════════════════════════════════════════════ -->
    <div class="tab-content active" id="__ID__-c0">
        <h2 class="seccion">Arquitectura Tecnológica</h2>
        <div class="graph-gallery">
            <div class="gallery-inner">
                <button class="gallery-btn prev" onclick="changeTechSlide__ID__(-1)">❮</button>
                <div class="graph-slide active tech-slide">
                    <div class="slide-card-container" style="font-size: 1.15em; line-height: 1.8;">
                        <h3 style="font-size: 1.9em; margin-bottom: 15px;">OpenRefine: Motor de Auditoría y ETL</h3>
                        <p>Herramienta <i>open-source</i> especializada en la depuración masiva de datos semiestructurados y heterogéneos. A diferencia de las hojas de cálculo tradicionales, OpenRefine opera mediante una arquitectura de servidor local orientada a bases de datos, lo que le permite procesar cientos de miles de registros sin bloqueos ni pérdida de rendimiento.</p>
                        <p>Su enfoque no destructivo garantiza que el dataset original nunca se sobrescriba. Las transformaciones se aplican sobre vistas virtuales facetadas, facilitando la identificación inmediata de valores atípicos, distribuciones anómalas y metadatos corruptos a través de sus potentes herramientas de perfilado estadístico.</p>
                        <p>En este proyecto, procesó fluidamente los <b>~130.000 registros</b> del catálogo global de <i>Wine Enthusiast</i>.</p>
                    </div>
                </div>
                <div class="graph-slide tech-slide">
                    <div class="slide-card-container" style="font-size: 1.15em; line-height: 1.8;">
                        <h3 style="font-size: 1.9em; margin-bottom: 15px;">GREL: General Refine Expression Language</h3>
                        <p>GREL es el poderoso lenguaje funcional nativo de OpenRefine, diseñado específicamente para la manipulación avanzada de cadenas de texto y el enriquecimiento de datos. Su sintaxis, inspirada en JavaScript pero optimizada para ETL, permite encadenar métodos (<i>method chaining</i>) para lograr transformaciones complejas en una sola línea de código.</p>
                        <p>Es la columna vertebral de este proyecto. Mediante GREL, no solo limpiamos datos, sino que hicimos <b>Feature Engineering</b>: extrajimos años de cosecha utilizando Expresiones Regulares (Regex) con límites de palabra (<code>\b</code>), categorizamos niveles de precio evaluando árboles lógicos anidados (<code>if/else</code>), y derivamos perfiles sensoriales completos filtrando matrices dinámicas de palabras clave.</p>
                        <p>Incluso el consumo de la <b>API externa de traducción</b> (HTTP Fetching) con sus respectivos delays y el parseo del JSON resultante, se ejecutaron nativa y exclusivamente con GREL.</p>
                    </div>
                </div>
                <div class="graph-slide tech-slide">
                    <div class="slide-card-container" style="font-size: 1.15em; line-height: 1.8;">
                        <h3 style="font-size: 1.9em; margin-bottom: 15px;">Clustering y Repositorio Reproducible</h3>
                        <p>La normalización de nombres comerciales provenientes de más de 12 países y múltiples idiomas es un desafío clásico de <i>Data Quality</i>. Para resolver la dispersión (ej: <i>Pinot Grigio</i> vs <i>Pinot Gris</i>), implementamos algoritmos de <b>Clustering de Texto</b>.</p>
                        <p>Utilizamos <b>Key Collision (Fingerprint)</b> para homogeneizar diferencias de puntuación, mayúsculas y orden de palabras, y <b>Nearest Neighbor</b> con métrica de distancia de <b>Levenshtein (radio 2.0)</b> para capturar errores de tipeo y variantes fonéticas cruzadas.</p>
                        <p><b>Reproducibilidad absoluta:</b> Cada uno de los pasos de limpieza, GREL y clustering quedó registrado secuencialmente en el archivo <code>pipeline_etl.json</code>. Este archivo funciona como un script ejecutable que permite aplicar exactamente el mismo pipeline sobre nuevos lotes de datos de forma automatizada.</p>
                    </div>
                </div>
                <button class="gallery-btn next" onclick="changeTechSlide__ID__(1)">❯</button>
            </div>
            <div class="pagination-dots" id="tech-dots-__ID__">
                <span class="dot active"></span><span class="dot"></span><span class="dot"></span>
            </div>
        </div>
    </div>

    <!-- ══════════════════════════════════════════════
         PESTAÑA 2 — El Dataset (5 slides con código)
    ══════════════════════════════════════════════ -->
    <div class="tab-content" id="__ID__-c1">
        <h2 class="seccion">Pipeline de Limpieza y Enriquecimiento</h2>
        <div class="graph-gallery">
            <div class="gallery-inner">
                <button class="gallery-btn prev" onclick="changeDataSlide__ID__(-1)">❮</button>

                <!-- SLIDE D1 · Depuración Estructural -->
                <div class="graph-slide active data-slide">
                    <div class="slide-card-container">
                        <h3>1. Depuración Estructural y Tipado</h3>
                        <p>El dataset de Kaggle incluía una columna residual <code>Unnamed: 0</code> generada automáticamente durante la exportación, sin ningún valor analítico. Se eliminó para reducir el peso del archivo.</p>
                        <p>Los campos <strong>points</strong> (rango 80–100) y <strong>price</strong> llegaron tipados como cadenas de texto. Sin conversión a número no es posible aplicar facetas de rango, calcular percentiles ni realizar correlaciones estadísticas.</p>
                        <div class="code-grid">
                            <div class="code-grid-col">
                                <div class="code-label">OpenRefine · Eliminar columna residual de Kaggle</div>
                                <div class="code-block"><span class="cm">// core/column-removal</span>
&#123;
  <span class="str">"op"</span>: <span class="str">"core/column-removal"</span>,
  <span class="str">"columnName"</span>: <span class="str">"Column"</span>
  <span class="cm">// alias "Unnamed: 0"</span>
&#125;</div>
                            </div>
                            <div class="code-grid-col">
                                <div class="code-label">GREL · Conversión a numérico (points y price)</div>
                                <div class="code-block"><span class="cm">// onError: keep-original</span>
<span class="cm">// → preserva nulos sin romper filas</span>

<span class="fn">value</span>.<span class="kw">toNumber</span>()

<span class="cm">// "90"  → 90   (int, puntuable)
// "24.0"→ 24.0 (float, USD)
// null  → null (conservado)</span></div>
                            </div>
                        </div>
                    </div>
                </div>

                <!-- SLIDE D2 · Extracción de Añadas -->
                <div class="graph-slide data-slide">
                    <div class="slide-card-container">
                        <h3>2. Extracción de Añadas (Regex)</h3>
                        <p>El año de cosecha no existía como campo explícito: estaba embebido en el <code>title</code> comercial de cada botella (ej: <em>"Château Margaux 2015 Grand Vin"</em>). Se creó la columna <code>vintage_year</code> mediante ingeniería de características.</p>
                        <p>Los límites de palabra (<code>\b</code>) evitan capturar números de barrica, códigos de lote y años de fundación de bodegas anteriores al siglo XX. Los espumantes <strong>Non-Vintage (NV)</strong> quedan como <code>null</code> sin propagar errores — estos nulos son luego capturados por la variable indicadora <code>is_non_vintage</code>.</p>
                        <div class="code-label">GREL · Nueva columna vintage_year — basada en title</div>
                        <div class="code-block"><span class="cm">// Captura años entre 1900 y 2029 con límites de palabra</span>
<span class="fn">grel:</span><span class="kw">with</span>(
  value.<span class="kw">match</span>(<span class="str">/.*?\\b(19\\d\\d|20[0-2]\\d)\\b.*/</span>),
  m,                          <span class="cm">// m = array de grupos capturados</span>
  <span class="kw">if</span>(<span class="fn">isNotNull</span>(m),
    m[<span class="num">0</span>].<span class="kw">toNumber</span>(),   <span class="cm">// → año numérico  (ej: 2015)</span>
    <span class="kw">null</span>               <span class="cm">// → null si es Non-Vintage</span>
  )
)</div>
                    </div>
                </div>

                <!-- SLIDE D3 · Clustering y Segmentación -->
                <div class="graph-slide data-slide">
                    <div class="slide-card-container">
                        <h3>3. Clustering y Segmentación Comercial</h3>
                        <p>El dataset combinaba convenciones lingüísticas de +12 países: la misma varietal aparecía como <em>Pinot Grigio</em> (italiano), <em>Pinot Gris</em> (francés) y <em>Pinot Gris</em> (inglés). Se aplicó <strong>Levenshtein (radio 2.0)</strong> y <strong>Key Collision Fingerprint</strong> para unificarlas. La columna <code>price_tier</code> gestiona los ~8.900 registros sin precio con un valor por defecto explícito, evitando que queden fuera de los cruces con facetas.</p>
                        <div class="code-grid">
                            <div class="code-grid-col">
                                <div class="code-label">OpenRefine · Clustering — ejemplos unificados</div>
                                <div class="code-block"><span class="cm">// Familia Pinot (italiano/francés/inglés)</span>
<span class="str">"Pinot Grigio"</span>, <span class="str">"Pinot Nero"</span>
    → <span class="str">"Pinot Gris"</span> / <span class="str">"Pinot Noir"</span>

<span class="cm">// Moscatel (5 variantes → 1)</span>
<span class="str">"Muscatel"</span>, <span class="str">"Moscadello"</span>
    → <span class="str">"Muscadelle"</span>

<span class="cm">// Germánicas</span>
<span class="str">"Blauburgunder"</span>, <span class="str">"Früburgunder"</span>
    → <span class="str">"Grauburgunder"</span></div>
                            </div>
                            <div class="code-grid-col">
                                <div class="code-label">GREL · Nueva columna price_tier</div>
                                <div class="code-block"><span class="cm">// ~8.900 nulos → valor por defecto</span>
<span class="fn">grel:</span><span class="kw">if</span>(<span class="fn">isBlank</span>(value),
  <span class="str">"Sin precio informado"</span>,

  <span class="kw">if</span>(value.<span class="kw">toNumber</span>() <span class="op">&lt;</span> <span class="num">20</span>,
    <span class="str">"1. Económico (&lt;$20)"</span>,

    <span class="kw">if</span>(value.<span class="kw">toNumber</span>() <span class="op">&lt;=</span> <span class="num">50</span>,
      <span class="str">"2. Gama Media ($20-$50)"</span>,
      <span class="str">"3. Premium (&gt;$50)"</span>
    )
  )
)</div>
                            </div>
                        </div>
                    </div>
                </div>

                <!-- SLIDE D4 · Enriquecimiento Sensorial y API -->
                <div class="graph-slide data-slide">
                    <div class="slide-card-container">
                        <h3>4. Enriquecimiento Sensorial y API REST</h3>
                        <p>Se clasificaron <strong>16 descriptores organolépticos</strong> mediante búsqueda de keywords en el texto libre de la descripción. Para evitar filas vacías, se asigna <em>"Equilibrado / Sutil"</em> como valor por defecto cualitativo para el 100% de los registros.</p>
                        <p>La traducción vía API se restringió a los <strong>19 vinos perfectos (100 pts)</strong> del mundo. Se usó solo la primera oración para cumplir el límite de 500 chars del endpoint gratuito, con <code>delay: 500ms</code> para respetar la cuota. La respuesta JSON se parseó extrayendo <code>responseData.translatedText</code>, que decodifica automáticamente entidades Unicode (ñ, á, é).</p>
                        <div class="code-grid">
                            <div class="code-grid-col">
                                <div class="code-label">GREL · perfil_sensorial — 16 keywords</div>
                                <div class="code-block"><span class="fn">grel:</span><span class="kw">with</span>(
  <span class="kw">filter</span>([
    <span class="kw">if</span>(value.<span class="fn">toLowercase</span>()
       .<span class="fn">contains</span>(<span class="str">"oak"</span>),      <span class="str">"Roble"</span>,   <span class="kw">null</span>),
    <span class="kw">if</span>(value.<span class="fn">toLowercase</span>()
       .<span class="fn">contains</span>(<span class="str">"tannin"</span>),   <span class="str">"Taninos"</span>,  <span class="kw">null</span>),
    <span class="kw">if</span>(value.<span class="fn">toLowercase</span>()
       .<span class="fn">contains</span>(<span class="str">"cherry"</span>),   <span class="str">"Cereza"</span>,   <span class="kw">null</span>),
    <span class="cm">/* ... 13 perfiles más ... */</span>
  ], v, <span class="fn">isNonBlank</span>(v)).<span class="kw">join</span>(<span class="str">", "</span>),
  tags,
  <span class="kw">if</span>(<span class="fn">isNonBlank</span>(tags), tags, <span class="str">"Equilibrado / Sutil"</span>)
)</div>
                            </div>
                            <div class="code-grid-col">
                                <div class="code-label">GREL · URL API MyMemory + parseo JSON</div>
                                <div class="code-block"><span class="cm">// Faceta: points = 100 (19 botellas)</span>
<span class="str">"https://api.mymemory.translated.net"</span>
<span class="op">+</span> <span class="str">"/get?q="</span>
<span class="op">+</span> value.<span class="fn">split</span>(<span class="str">". "</span>)[<span class="num">0</span>]   <span class="cm">// 1ª oración</span>
        .<span class="kw">escape</span>(<span class="str">"url"</span>)    <span class="cm">// RFC 3986</span>
<span class="op">+</span> <span class="str">"&amp;langpair=en%7Ces"</span>  <span class="cm">// pipe → %7C</span>
<span class="cm">// delay: 500ms · cache: true

// Parseo de respuesta:
</span><span class="fn">value</span>.<span class="kw">parseJson</span>()
    .responseData
    .translatedText  <span class="cm">// → "Notas de cereza..."</span></div>
                            </div>
                        </div>
                    </div>
                </div>

                <!-- SLIDE D5 · Feature Engineering y Calidad de Datos -->
                <div class="graph-slide data-slide">
                    <div class="slide-card-container">
                        <h3>5. Ingeniería de Variables y Calidad de Datos</h3>
                        <p>En la segunda sesión del pipeline se eliminó <code>taster_twitter_handle</code> (colineal con <code>taster_name</code>, sin varianza explicativa adicional). Se imputaron ~30.000 nulos en <code>designation</code> y los vacíos geográficos de <code>region_1</code> y <code>region_2</code> para evitar errores en visualizaciones y modelos.</p>
                        <p>Para preservar el tipo numérico de <code>vintage_year</code>, en lugar de inyectar texto en una columna de enteros continuos, se creó la variable indicadora binaria <code>is_non_vintage</code> (0/1) que permite que los algoritmos de regresión capturen el efecto estadístico de no tener añada.</p>
                        <div class="code-grid">
                            <div class="code-grid-col">
                                <div class="code-label">GREL · Imputación de nulos estructurales</div>
                                <div class="code-block"><span class="cm">// designation: ~30.000 nulos
// (vinos estándar sin viñedo específico)</span>
<span class="kw">if</span>(<span class="fn">isBlank</span>(value),
  <span class="str">"Sin designación"</span>, value)

<span class="cm">// taster_name: reseñas sin autor</span>
<span class="kw">if</span>(<span class="fn">isBlank</span>(value),
  <span class="str">"Catador anónimo"</span>, value)

<span class="cm">// region_1 / region_2 (geografía US)</span>
<span class="kw">if</span>(<span class="fn">isBlank</span>(value),
  <span class="str">"Sin región"</span>, value)</div>
                            </div>
                            <div class="code-grid-col">
                                <div class="code-label">GREL · Variable dummy is_non_vintage (ML)</div>
                                <div class="code-block"><span class="cm">// Paso 1: crear indicador binario
// (nulo en vintage_year = Non-Vintage)</span>
<span class="kw">if</span>(<span class="fn">isBlank</span>(value), <span class="num">1</span>, <span class="num">0</span>)

<span class="cm">// Paso 2: corregir columna original
// (preservar enteros, sin texto)</span>
<span class="kw">if</span>(<span class="fn">isBlank</span>(value), <span class="num">0</span>, value)

<span class="cm">// Resultado:
// vintage_year: 2015 / 2008 / 0
// is_non_vintage: 0   / 0   / 1</span></div>
                            </div>
                        </div>
                    </div>
                </div>

                <button class="gallery-btn next" onclick="changeDataSlide__ID__(1)">❯</button>
            </div>
            <div class="pagination-dots" id="data-dots-__ID__">
                <span class="dot active"></span><span class="dot"></span><span class="dot"></span>
                <span class="dot"></span><span class="dot"></span>
            </div>
        </div>
    </div>

    <!-- ══════════════════════════════════════════════
         PESTAÑA 3 — Análisis Visual (6 slides)
    ══════════════════════════════════════════════ -->
    <div class="tab-content" id="__ID__-c2">
        <h2 class="seccion">Exploración Interactiva de Datos</h2>
        <div class="graph-gallery">
            <div class="gallery-inner">
                <button class="gallery-btn prev" onclick="changeGraphSlide__ID__(-1)">❮</button>

                <!-- G1: Boxplot -->
                <div class="graph-slide active graph-slide-item">
                    <div class="grafico-box grande"><iframe srcdoc="{src1}" scrolling="no"></iframe></div>
                    <div class="graph-desc">
                        <h3>Distribución de Calidad por Segmento de Precio</h3>
                        <p>El gráfico de cajas muestra cómo varía la mediana y la dispersión del puntaje a medida que escalamos de gama Económica a Premium. La columna <code>price_tier</code> fue construida en OpenRefine con GREL.</p>
                    </div>
                </div>

                <!-- G2: Barras sensoriales -->
                <div class="graph-slide graph-slide-item">
                    <div class="grafico-box grande"><iframe src="{src2}" scrolling="no"></iframe></div>
                    <div class="graph-desc">
                        <h3>Top Descriptores Sensoriales</h3>
                        <p>Frecuencia de los 10 perfiles organolépticos más encontrados en las notas de cata. Extraídos de la columna <code>perfil_sensorial</code> construida con 16 keywords en GREL sobre las ~130.000 descripciones.</p>
                    </div>
                </div>

                <!-- G3: Scatter precio/puntaje -->
                <div class="graph-slide graph-slide-item">
                    <div class="grafico-box grande"><iframe src="{src3}" scrolling="no"></iframe></div>
                    <div class="graph-desc">
                        <h3>Value for Money: Precio vs Puntaje</h3>
                        <p>Diagrama de dispersión que expone la relación entre costo y calidad percibida, segmentado por gama comercial. Los outliers de alto precio con puntaje moderado son visibles como oportunidades de análisis.</p>
                    </div>
                </div>

                <!-- G4: Mapa Francia -->
                <div class="graph-slide graph-slide-item">
                    <div class="grafico-box grande"><iframe src="{src4}" scrolling="no"></iframe></div>
                    <div class="graph-desc">
                        <h3>Mapa de Burbujas — Regiones Vitivinícolas de Francia</h3>
                        <p>El <strong>tamaño de la burbuja</strong> representa la cantidad de vinos de la región y el <strong>color</strong> (rosa → dorado) la puntuación media. Bordeaux y Burgundy destacan por volumen; Champagne lidera en calidad.</p>
                    </div>
                </div>

                <!-- G5: Mapa Italia -->
                <div class="graph-slide graph-slide-item">
                    <div class="grafico-box grande"><iframe src="{src5}" scrolling="no"></iframe></div>
                    <div class="graph-desc">
                        <h3>Mapa de Burbujas — Regiones Vitivinícolas de Italia</h3>
                        <p>Tuscany concentra el mayor volumen del dataset italiano con casi 6.000 reseñas. Piedmont y Veneto le siguen en cantidad, mientras que el color dorado señala las regiones de mayor puntuación promedio.</p>
                    </div>
                </div>

                <!-- G6: Mapa EE.UU. -->
                <div class="graph-slide graph-slide-item">
                    <div class="grafico-box grande"><iframe src="{src6}" scrolling="no"></iframe></div>
                    <div class="graph-desc">
                        <h3>Mapa Coroplético — Calidad Vitivinícola por Estado (EE. UU.)</h3>
                        <p>EE. UU. es el país más representado del dataset. El color más dorado indica mayor puntuación promedio por estado. California concentra el mayor volumen de reseñas, mientras que estados del Noroeste como Oregon muestran puntuaciones destacadas.</p>
                    </div>
                </div>

                <button class="gallery-btn next" onclick="changeGraphSlide__ID__(1)">❯</button>
            </div>
            <div class="pagination-dots" id="graph-dots-__ID__">
                <span class="dot active"></span><span class="dot"></span><span class="dot"></span>
                <span class="dot"></span><span class="dot"></span><span class="dot"></span>
            </div>
        </div>
    </div>
</div>

<script>
var root__ID__ = document.getElementById("__ID__");
var fsBtnText__ID__ = document.getElementById("__ID__-fs-text");

function changeTechSlide__ID__(d) {{
    let s = root__ID__.querySelectorAll('.tech-slide');
    let dots = root__ID__.querySelectorAll('#tech-dots-__ID__ .dot');
    let c = Array.from(s).findIndex(el => el.classList.contains('active'));
    s[c].classList.remove('active'); dots[c].classList.remove('active');
    let next = (c + d + s.length) % s.length;
    s[next].classList.add('active'); dots[next].classList.add('active');
}}
function changeDataSlide__ID__(d) {{
    let s = root__ID__.querySelectorAll('.data-slide');
    let dots = root__ID__.querySelectorAll('#data-dots-__ID__ .dot');
    let c = Array.from(s).findIndex(el => el.classList.contains('active'));
    s[c].classList.remove('active'); dots[c].classList.remove('active');
    let next = (c + d + s.length) % s.length;
    s[next].classList.add('active'); dots[next].classList.add('active');
}}
function changeGraphSlide__ID__(d) {{
    let s = root__ID__.querySelectorAll('.graph-slide-item');
    let dots = root__ID__.querySelectorAll('#graph-dots-__ID__ .dot');
    let c = Array.from(s).findIndex(el => el.classList.contains('active'));
    s[c].classList.remove('active'); dots[c].classList.remove('active');
    let next = (c + d + s.length) % s.length;
    s[next].classList.add('active'); dots[next].classList.add('active');
}}

const galeria = [
    "https://sakwinetravel.com/wp-content/uploads/2018/07/tm0003-bodegas-uco2.jpg.webp",
    "https://dynamic-media-cdn.tripadvisor.com/media/photo-o/06/f7/6b/14/bodegas-caro.jpg?w=1200&h=-1&s=1",
    "https://www.amerian.com/wp-content/uploads/2025/11/zaca.jpg",
    "https://gower.ar/pt/wp-content/uploads/2021/09/Blog-Valle-de-Uco.jpg",
    "https://imagenes2.eltiempo.com/files/image_1200_535/uploads/2023/07/13/64b022c02dced.jpeg",
    "https://media.dondeir.com/2021/11/vinedos-con-hotel.jpg",
    "https://i.pinimg.com/originals/8a/06/fb/8a06fb84f54d0c33a0bae4ec8c08e95a.jpg",
    "https://media.istockphoto.com/id/2148938420/es/foto/varias-botellas-de-vino-tinto-negro-con-etiquetas-rojas-alineadas-y-apiladas-horizontalmente.jpg?s=612x612&w=0&k=20&c=oxIi6GQNtCE48J8RbPzpvOYKxJmI6mewyBxfYsVgSEU=",
    "https://images.ecestaticos.com/ZVPLmxQlj3l3-VCPFhx2j2PXqv0=/0x0:1280x847/1200x900/filters:fill(white):format(jpg)/f.elconfidencial.com%2Foriginal%2Fd65%2Facd%2F953%2Fd65acd953573736e49e2034e0896a3ce.jpg",
    "https://escuelaversailles.com/wp-content/uploads/tipos-de-vino.jpg",
    "https://casadevinosmendoza.com.ar/wp-content/uploads/2025/10/Vinos-para-asados-y-juntadas.png"
];
let shuffled = galeria.sort(() => 0.5 - Math.random()).slice(0, 6);
const introCar = document.getElementById("__ID__-intro-carousel");
const mainCar  = document.getElementById("__ID__-main-carousel");
shuffled.forEach((src, idx) => {{
    let i1 = document.createElement('img'); i1.src = src;
    if(idx === 0) i1.classList.add('active'); introCar.appendChild(i1);
    let i2 = document.createElement('img'); i2.src = src;
    if(idx === 0) i2.classList.add('active'); mainCar.appendChild(i2);
}});
setInterval(() => {{
    let s = introCar.querySelectorAll('img');
    let c = Array.from(s).findIndex(el => el.classList.contains('active'));
    if(c >= 0) {{ s[c].classList.remove('active'); s[(c+1)%s.length].classList.add('active'); }}
}}, 2500);
setInterval(() => {{
    let s = mainCar.querySelectorAll('img');
    let c = Array.from(s).findIndex(el => el.classList.contains('active'));
    if(c >= 0) {{ s[c].classList.remove('active'); s[(c+1)%s.length].classList.add('active'); }}
}}, 4500);

function checkFullscreen() {{
    if(fsBtnText__ID__) {{
        fsBtnText__ID__.innerText = (document.fullscreenElement || window.innerHeight === screen.height)
            ? "Minimizar pantalla" : "Maximizar pantalla";
    }}
}}
document.addEventListener("fullscreenchange", checkFullscreen);
window.addEventListener("resize", checkFullscreen);

function iniciarPresentacion__ID__() {{
    let btnBox = root__ID__.querySelector('#__ID__-btnBox');
    btnBox.style.opacity = 0; setTimeout(() => btnBox.style.display = 'none', 400);
    let content = root__ID__.querySelector('#__ID__-introContent');
    content.style.display = 'flex';
    let members = root__ID__.querySelectorAll('#__ID__-intro .member');
    members.forEach((m, i) => {{
        setTimeout(() => {{ m.style.opacity = '1'; m.style.transform = 'translateY(0)'; }}, i * 500 + 200);
    }});
    setTimeout(() => {{
        let stack = root__ID__.querySelector('#__ID__-memberStack'); stack.classList.add('crushed');
        setTimeout(() => {{
            let trubo = root__ID__.querySelector('#__ID__-trubo-presenta');
            trubo.style.display = 'block'; setTimeout(() => {{ trubo.style.opacity = 1; }}, 50);
            setTimeout(() => {{
                trubo.style.opacity = 0;
                setTimeout(() => {{
                    trubo.style.display = 'none';
                    let titleDiv = root__ID__.querySelector('#__ID__-introTitle');
                    titleDiv.style.display = 'block';
                    setTimeout(() => {{ titleDiv.classList.add('show'); }}, 50);
                    setTimeout(() => {{
                        let intro = root__ID__.querySelector('#__ID__-intro');
                        let mainContent = root__ID__.querySelector('#__ID__-main');
                        intro.style.opacity = 0; mainContent.style.display = 'block';
                        setTimeout(() => {{ mainContent.style.opacity = 1; intro.style.display = 'none'; }}, 50);
                    }}, 3500);
                }}, 1000);
            }}, 2500);
        }}, 500);
    }}, members.length * 500 + 1500);
}}

var tabs__ID__ = root__ID__.querySelector(".tabs").querySelectorAll("button");
var contents__ID__ = root__ID__.querySelectorAll(".tab-content");
function openTab__ID__(index) {{
    tabs__ID__.forEach((btn, i) => {{ if(i === index) btn.classList.add("on"); else btn.classList.remove("on"); }});
    contents__ID__.forEach((c, i) => {{ if(i === index) c.classList.add("active"); else c.classList.remove("active"); }});
}}

document.getElementById("__ID__-fs").onclick = function() {{
    var ok = function() {{
        var clonedNode = root__ID__.cloneNode(true);
        var introClon = clonedNode.querySelector('#__ID__-intro');
        if(root__ID__.querySelector('#__ID__-intro').style.display === 'none') {{ introClon.style.display = 'none'; }}
        var styleOverride = document.createElement('style');
        styleOverride.innerHTML = '.gx {{ height: 100vh !important; width: 100vw !important; max-width: none !important; border-radius: 0 !important; }}';
        var container = document.createElement('div'); container.appendChild(styleOverride); container.appendChild(clonedNode);
        var extraScript = "<script>" +
        "function changeTechSlide__ID__(d){{ let s=document.querySelectorAll('.tech-slide'); let dots=document.querySelectorAll('#tech-dots-__ID__ .dot'); let c=Array.from(s).findIndex(el=>el.classList.contains('active')); s[c].classList.remove('active'); dots[c].classList.remove('active'); let next=(c+d+s.length)%s.length; s[next].classList.add('active'); dots[next].classList.add('active'); }}" +
        "function changeDataSlide__ID__(d){{ let s=document.querySelectorAll('.data-slide'); let dots=document.querySelectorAll('#data-dots-__ID__ .dot'); let c=Array.from(s).findIndex(el=>el.classList.contains('active')); s[c].classList.remove('active'); dots[c].classList.remove('active'); let next=(c+d+s.length)%s.length; s[next].classList.add('active'); dots[next].classList.add('active'); }}" +
        "function changeGraphSlide__ID__(d){{ let s=document.querySelectorAll('.graph-slide-item'); let dots=document.querySelectorAll('#graph-dots-__ID__ .dot'); let c=Array.from(s).findIndex(el=>el.classList.contains('active')); s[c].classList.remove('active'); dots[c].classList.remove('active'); let next=(c+d+s.length)%s.length; s[next].classList.add('active'); dots[next].classList.add('active'); }}" +
        "document.getElementById('__ID__-fs').onclick=function(){{ if(document.fullscreenElement){{document.exitFullscreen();}}else{{document.documentElement.requestFullscreen();}} }};" +
        "document.addEventListener('fullscreenchange',function(){{ var t=document.getElementById('__ID__-fs-text'); if(t){{t.innerText=document.fullscreenElement?'Minimizar pantalla':'Maximizar pantalla';}} }});" +
        "function openTab__ID__(i){{ document.querySelectorAll('.tabs button').forEach((b,j)=>{{if(j===i)b.classList.add('on');else b.classList.remove('on');}}); document.querySelectorAll('.tab-content').forEach((c,j)=>{{if(j===i)c.classList.add('active');else c.classList.remove('active');}}); }}" +
        "<\\/script>";
        var b = new Blob(["<!doctype html><meta charset='utf-8'><body style='margin:0;padding:0;background:#fcf9f2;overflow-x:hidden;'>" + container.innerHTML + extraScript + "</body>"], {{type:"text/html"}});
        var w = window.open(URL.createObjectURL(b), "_blank");
        if(!w) alert("Colab bloqueó la ventana emergente. Permite las ventanas emergentes en tu navegador.");
    }};
    if(document.fullscreenElement) {{ document.exitFullscreen(); }}
    else {{
        try {{
            var r = root__ID__.requestFullscreen && root__ID__.requestFullscreen();
            if(r && r.catch) r.catch(ok); else if(!r) ok();
        }} catch(e) {{ ok(); }}
    }}
}};
</script>
</div>
</body>
</html>
'''

_pagina = _html.replace('__ID__', 'gx' + uuid.uuid4().hex[:6])
display(HTML(_pagina))


In [ ]:
# @title
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# @title
import subprocess, sys, os, time, importlib
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "catboost"])
import numpy as np, pandas as pd, joblib
from google.colab import drive
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import TargetEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import confusion_matrix
from catboost import CatBoostRegressor


CSV = "/content/drive/MyDrive/PF/winemag-data-130k-v2-csv (1).csv"

SEED = 42
os.makedirs(SALIDA, exist_ok=True)

# ---------------------------------------------------------------- Funciones de variables (compartidas con la web)
UTIL = r'''import numpy as np, pandas as pd
CATEGORICAS=["country","province","region_1","region_2","variety","designation","winery"]
NUM=["precio_faltante","vintage_year","is_non_vintage","lp","winery_precio","winery_variedades",
 "precio_rel_variety","precio_rel_province","precio_rel_country","precio_pct_pais",
 "winery_n","variety_n","province_n","region_1_n","designation_n",
 "des_reserva","des_cru","des_vv","des_finca","des_espum","sin_designacion","sin_region"]
FEATURES=CATEGORICAS+NUM
CLAVES={"reserva":"reserv|riserva|réserve","cru":"grand cru|premier cru|cru classé","vv":"vieill|old vine|vinhas velhas|alte reben","finca":"estate|single|vineyard|finca|vigna|block","espum":"brut|extra dry|cuvée|champagne"}
def crear_ref(df):
    t=df.assign(lp=np.log(df["price"]))
    return {"freq":{c:df[c].value_counts().to_dict() for c in ["winery","variety","province","region_1","designation"]},
     "w_precio":t.groupby("winery")["lp"].mean().to_dict(),"w_var":df.groupby("winery")["variety"].nunique().to_dict(),
     "med":{c:t.groupby(c)["lp"].median().to_dict() for c in ["variety","province","country"]},
     "lp_sorted":{k:np.sort(g.dropna().values) for k,g in t.groupby("country")["lp"]},
     "mediana_anio":float(df["vintage_year"].median())}
def construir(d,ref):
    d=d.copy()
    d["lp"]=np.log(d["price"].astype(float))
    d["precio_faltante"]=d["lp"].isna().astype(int)
    d["vintage_year"]=d["vintage_year"].astype(float).fillna(ref["mediana_anio"]).clip(1990,2017)
    for c,m in ref["freq"].items(): d[c+"_n"]=d[c].map(m).fillna(1)
    d["winery_precio"]=d["winery"].map(ref["w_precio"])
    d["winery_variedades"]=d["winery"].map(ref["w_var"]).fillna(1)
    for c in ["variety","province","country"]:
        d["precio_rel_"+c]=d["lp"]-d[c].map(ref["med"][c])
    def pct(l,p):
        a=ref["lp_sorted"].get(p)
        if a is None or len(a)==0 or np.isnan(l): return np.nan
        return (np.searchsorted(a,l,"left")+np.searchsorted(a,l,"right"))/2/len(a)
    d["precio_pct_pais"]=[pct(l,p) for l,p in zip(d["lp"],d["country"])]
    des=d["designation"].fillna("").str.lower()
    for k,pat in CLAVES.items(): d["des_"+k]=des.str.contains(pat).astype(int)
    d["sin_designacion"]=(d["designation"]=="Sin designación").astype(int)
    d["sin_region"]=(d["region_1"]=="Sin región").astype(int)
    return d[FEATURES]
'''
with open(f"{SALIDA}/utilidades.py", "w", encoding="utf-8") as f:
    f.write(UTIL)
sys.path.insert(0, SALIDA)
import utilidades; importlib.reload(utilidades)
from utilidades import CATEGORICAS, NUM, crear_ref, construir

# ---------------------------------------------------------------- Datos
PAISES = ["Argentina", "Armenia", "Australia", "Austria", "Bosnia and Herzegovina", "Brazil", "Bulgaria",
          "Canada", "Chile", "China", "Croatia", "Cyprus", "Czech Republic", "Egypt", "England", "France",
          "Georgia", "Germany", "Greece", "Hungary", "India", "Israel", "Italy", "Lebanon", "Luxembourg",
          "Macedonia", "Mexico", "Moldova", "Morocco", "New Zealand", "Null", "Peru", "Portugal", "Romania",
          "Serbia", "Slovakia", "Slovenia", "South Africa", "Spain", "Switzerland", "Turkey", "Ukraine",
          "Uruguay", "US"]
df = pd.read_csv(CSV)

# Solución al KeyError: Si 'country' ya está en el DataFrame, no intentamos reconstruirla de las columnas binarias
if "country" not in df.columns:
    df["country"] = [PAISES[i] for i in df[[f"country_binary {i}" for i in range(1, 45)]].values.argmax(axis=1)]

for c in CATEGORICAS:
    df[c] = df[c].fillna("Sin dato")

ref = crear_ref(df)
X, y = construir(df, ref), df["points"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=SEED)

def a_grupo(p):
    p = np.asarray(p)
    return np.where(p <= 86, 0, np.where(p <= 90, 1, 2))

# ---------------------------------------------------------------- Modelos
def nuevo_hgb():
    prep = ColumnTransformer([
        ("cat", TargetEncoder(target_type="continuous", smooth="auto", cv=5, random_state=SEED), CATEGORICAS),
        ("num", "passthrough", NUM)])
    m = HistGradientBoostingRegressor(max_iter=1500, learning_rate=0.03, max_leaf_nodes=24, min_samples_leaf=40,
                                      l2_regularization=1.0, early_stopping=True, validation_fraction=0.1,
                                      n_iter_no_change=40, random_state=SEED)
    return Pipeline([("prep", prep), ("modelo", m)])

def nuevo_cb():
    return CatBoostRegressor(iterations=1000, learning_rate=0.06, depth=7, random_seed=SEED,
                             verbose=0, cat_features=CATEGORICAS)

# ---------------------------------------------------------------- Evaluación (80/20)
t0 = time.time()
hgb, cb = nuevo_hgb().fit(Xtr, ytr), nuevo_cb().fit(Xtr, ytr)
preds = {"HGB": hgb.predict(Xte), "CatBoost": cb.predict(Xte)}
preds["Ensamble"] = (preds["HGB" ] + preds["CatBoost"]) / 2

filas = []
for nombre, p in preds.items():
    p = np.clip(p, 80, 100)
    err = np.round(p) - yte
    cm = confusion_matrix(a_grupo(yte), a_grupo(np.round(p)), labels=[0, 1, 2])
    filas.append({"Modelo": nombre, "Exacto (%)": (err == 0).mean() * 100, "±1 (%)": (np.abs(err) <= 1).mean() * 100,
                  "±2 (%)": (np.abs(err) <= 2).mean() * 100, "Grupo acertado (%)": np.mean(a_grupo(np.round(p)) == a_grupo(yte)) * 100,
                  "MAE": np.abs(p - yte).mean(), "Falsos excelentes": int(cm[0, 2]), "Joyas ocultas": int(cm[2, 0])})
res = pd.DataFrame(filas).set_index("Modelo").round(2)
print(res.to_string())
res.to_csv(f"{SALIDA}/resultados.csv")
sigma = float(np.std(np.clip(preds["Ensamble"], 80, 100) - yte))   # error típico, usado para estimar probabilidades
print(f"Error típico (desvío): {sigma:.2f} puntos  |  tiempo evaluación: {time.time()-t0:.0f}s")

# ---------------------------------------------------------------- Modelo final con el 100 % de los datos
hgb_f, cb_f = nuevo_hgb().fit(X, y), nuevo_cb().fit(X, y)

# ---------------------------------------------------------------- Listas para los desplegables de la web
listas = {
    "paises": sorted(set(df["country"]) - {"Null"}),
    "provincias": {c: sorted(g["province"].unique()) for c, g in df.groupby("country")},
    "regiones": {p: sorted(g["region_1"].unique()) for p, g in df.groupby("province")},
    "bodegas": {f"{p}||{r}": sorted(g["winery"].unique()) for (p, r), g in df.groupby(["province", "region_1"])},
    "variedades": {c: list(g["variety"].value_counts().index) for c, g in df.groupby("country")},
    "designaciones": {w: sorted(set(g["designation"]) | {"Sin designación"}) for w, g in df.groupby("winery")},
    "region2": {f"{p}||{r}": g["region_2"].mode().iloc[0] for (p, r), g in df.groupby(["province", "region_1"])},
}

joblib.dump(hgb_f, f"{SALIDA}/hgb.joblib", compress=3)
cb_f.save_model(f"{SALIDA}/catboost.cbm")
joblib.dump({"ref": ref, "sigma": sigma, "listas": listas}, f"{SALIDA}/referencias.joblib", compress=3)
print("\nArchivos guardados en", SALIDA, ":", os.listdir(SALIDA))

          Exacto (%)  ±1 (%)  ±2 (%)  Grupo acertado (%)   MAE  Falsos excelentes  Joyas ocultas
Modelo                                                                                          
HGB            21.65   57.92   80.80               66.88  1.54                 62             47
CatBoost       20.97   57.43   80.55               66.24  1.55                 75             44
Ensamble       21.43   58.29   80.91               67.05  1.53                 70             42
Error típico (desvío): 1.96 puntos  |  tiempo evaluación: 285s

Archivos guardados en /content/drive/MyDrive/PF/modelos_vino : ['utilidades.py', '__pycache__', 'resultados.csv', 'hgb.joblib', 'catboost.cbm', 'referencias.joblib']


**WEB**

In [ ]:
# @title
import subprocess, sys, math
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio", "catboost"])
import numpy as np, pandas as pd, joblib, gradio as gr
from google.colab import drive
from catboost import CatBoostRegressor


SALIDA = "/content/drive/MyDrive/PF/modelos_vino"
sys.path.insert(0, SALIDA)
from utilidades import construir

hgb = joblib.load(f"{SALIDA}/hgb.joblib")
cb = CatBoostRegressor().load_model(f"{SALIDA}/catboost.cbm")
R = joblib.load(f"{SALIDA}/referencias.joblib")
ref, sigma, L = R["ref"], R["sigma"], R["listas"]

# ---------------------------------------------------------------- Predicción
GRUPOS = [("Regular", "86 puntos o menos", "#a0722a"), ("Bueno", "87 a 90 puntos", "#2f5d7c"), ("Excelente", "91 puntos o más", "#2f6b4a")]

def predecir(pais, prov, reg, bod, var, des, precio, anio):
    if not all([pais, prov, reg, bod, var, des, precio, anio]):
        return "<div class='vacio'>Complete todos los campos para obtener la estimación.</div>"
    fila = pd.DataFrame([{
        "country": pais, "province": prov, "region_1": reg, "region_2": L["region2"].get(f"{prov}||{reg}", "Sin subregión"),
        "variety": var, "designation": des, "winery": bod,
        "price": np.nan if precio == "na" else float(precio),
        "vintage_year": np.nan if anio == "nv" else float(anio), "is_non_vintage": 1 if anio == "nv" else 0}])
    X = construir(fila, ref)
    p = float(np.clip((hgb.predict(X)[0] + cb.predict(X)[0]) / 2, 80, 100))
    pt = int(round(p))
    g = 0 if pt <= 86 else (1 if pt <= 90 else 2)
    cdf = lambda x: 0.5 * (1 + math.erf((x - p) / (sigma * math.sqrt(2))))
    pr = [cdf(86.5), cdf(90.5) - cdf(86.5), 1 - cdf(90.5)]
    filas = "".join(
        f"<div class='fila'><span>{GRUPOS[i][0]}</span><div class='pista'><div style='width:{pr[i]*100:.0f}%;background:{GRUPOS[i][2]}'></div></div><b>{pr[i]*100:.0f} %</b></div>"
        for i in range(3))
    escala = (f"<div class='escala'><i style='width:32.5%;background:{GRUPOS[0][2]}'></i><i style='width:20%;background:{GRUPOS[1][2]}'></i>"
              f"<i style='width:47.5%;background:{GRUPOS[2][2]}'></i><u style='left:{(p-80)/20*100:.1f}%'></u></div>"
              "<div class='marcas'><span>80</span><span>86</span><span>90</span><span>100</span></div>")
    return f"""<div class='res'>
      <div class='et'>PUNTUACIÓN ESTIMADA</div><div class='num'>{p:.1f}</div>
      <div class='cat' style='background:{GRUPOS[g][2]}'>{GRUPOS[g][0]}</div><div class='et'>{GRUPOS[g][1]}</div>
      {escala}
      <div class='et' style='margin-top:6px'>Rango probable (±2 puntos): {max(80, p-2):.1f} a {min(100, p+2):.1f}</div>
      <div class='sec'>PROBABILIDAD ESTIMADA POR GRUPO</div>{filas}
      <p class='nota'>Estimación basada en datos previos a la cata (origen, bodega, variedad, designación, precio y añada). No sustituye la evaluación de un catador.</p></div>"""

# ---------------------------------------------------------------- Opciones
PRECIOS = [5, 7, 10, 12, 15, 18, 20, 25, 30, 35, 40, 50, 60, 75, 100, 150, 200, 300, 500]
OP_PRECIO = [(f"US$ {p}", str(p)) for p in PRECIOS] + [("No informado", "na")]
OP_ANIO = [(str(a), str(a)) for a in range(2017, 1989, -1)] + [("Sin añada", "nv")]

def vacio(): return gr.update(choices=[], value=None)
def on_pais(c):
    v = L["variedades"].get(c, [])
    return gr.update(choices=L["provincias"].get(c, []), value=None), vacio(), vacio(), gr.update(choices=v, value=None), vacio()
def on_prov(p): return gr.update(choices=L["regiones"].get(p, []), value=None), vacio(), vacio()
def on_reg(p, r): return gr.update(choices=L["bodegas"].get(f"{p}||{r}", []), value=None), vacio()
def on_bod(b):
    if not b: return vacio()
    d = L["designaciones"].get(b, ["Sin designación"])
    return gr.update(choices=d, value="Sin designación" if "Sin designación" in d else None)

CSS = """
:root,.dark,.gradio-container,.dark .gradio-container{color-scheme:light;
 --body-background-fill:#f4f1ec !important;--background-fill-primary:#ffffff !important;--background-fill-secondary:#f4f1ec !important;
 --block-background-fill:#ffffff !important;--block-border-color:#d9d4cb !important;--border-color-primary:#d9d4cb !important;
 --input-background-fill:#ffffff !important;--input-border-color:#cfc9bf !important;--input-background-fill-focus:#ffffff !important;
 --body-text-color:#1f2430 !important;--body-text-color-subdued:#6a7080 !important;--block-label-text-color:#4a5060 !important;
 --block-title-text-color:#4a5060 !important;--block-label-background-fill:transparent !important;--shadow-drop:none !important}
body,.gradio-container{background:#f4f1ec !important;color:#1f2430 !important;font-family:'Helvetica Neue',Arial,sans-serif !important}
.gradio-container{max-width:1040px !important;margin:0 auto !important;padding:0 20px 30px !important}
footer{display:none !important}
.cab{text-align:center;padding:34px 10px 22px;border-bottom:1px solid #d9d4cb;margin-bottom:26px}
.cab .sup{font-size:12px;letter-spacing:.18em;color:#6b1f2f;font-weight:600}
.cab h1{font:600 34px Georgia,'Times New Roman',serif;margin:8px 0 8px;color:#1f2430}
.cab p{margin:0 auto;max-width:640px;color:#6a7080;line-height:1.55}
.panel{background:#fff !important;border:1px solid #d9d4cb !important;border-radius:6px !important;padding:24px !important;gap:14px !important}
.panel .block{border:0 !important;box-shadow:none !important;background:transparent !important;padding:0 !important}
.titulo{font:600 12px Arial,sans-serif;letter-spacing:.12em;color:#6a7080;text-transform:uppercase;margin:0 0 6px;padding-bottom:10px;border-bottom:1px solid #e7e3dc}
#btn{background:#6b1f2f !important;color:#fff !important;border:0 !important;border-radius:4px !important;font-weight:600 !important;letter-spacing:.05em;padding:12px !important;margin-top:8px}
#btn:hover{background:#561726 !important}
.res{text-align:center;padding:4px 0}.et{color:#6a7080;font-size:12.5px;letter-spacing:.04em}
.num{font:600 72px Georgia,serif;line-height:1.1;margin:6px 0 10px;color:#1f2430}
.cat{display:inline-block;color:#fff;font-weight:600;padding:6px 18px;border-radius:3px;margin-bottom:6px}
.escala{position:relative;display:flex;height:10px;margin:22px 4px 6px;border-radius:2px}.escala i{display:block;height:100%}
.escala u{position:absolute;top:-6px;width:3px;height:22px;background:#1f2430;text-decoration:none}
.marcas{display:flex;justify-content:space-between;font-size:11px;color:#6a7080;margin:0 4px}
.sec{font:600 12px Arial,sans-serif;letter-spacing:.12em;color:#6a7080;margin:22px 0 8px;padding-top:14px;border-top:1px solid #e7e3dc}
.fila{display:flex;align-items:center;gap:10px;margin:7px 0;font-size:13px;text-align:left}.fila span{width:72px}.fila b{width:46px;text-align:right}
.pista{flex:1;height:8px;background:#e7e3dc;border-radius:2px;overflow:hidden}.pista div{height:100%}
.nota{font-size:12.5px;color:#6a7080;border-top:1px solid #e7e3dc;margin-top:20px;padding-top:12px;text-align:left}
.vacio{color:#6a7080;text-align:center;padding:60px 10px;line-height:1.6}
.pie{text-align:center;border-top:1px solid #d9d4cb;margin-top:30px;padding-top:20px;font-size:13px;color:#6a7080;line-height:1.7}
.pie .eq{display:flex;justify-content:center;gap:28px;flex-wrap:wrap;margin-top:12px;color:#1f2430}.ln{font-weight:700}
"""
PIE = """<div class='pie'><div>Sitio desarrollado con la asistencia de Claude (Anthropic). Modelos entrenados con el dataset Wine Reviews, depurado con OpenRefine.</div>
<div class='eq'><div class="member"><span class="ln">CAMPOS</span>, Fiorella</div><div class="member"><span class="ln">MARECO</span>, Carlos</div>
<div class="member"><span class="ln">SALVATORE</span>, Santiago</div><div class="member"><span class="ln">SITTNER</span>, Germán</div></div></div>"""

with gr.Blocks(css=CSS, title="Estimador de Calidad de Vinos", theme=gr.themes.Base()) as demo:
    gr.HTML("<div class='cab'><div class='sup'>ANÁLISIS DE CALIDAD ENOLÓGICA</div><h1>Estimador de Calidad de Vinos</h1><p>Filtro comercial que estima la puntuación (80 a 100) y el grupo de calidad de un vino a partir de datos previos a la cata, sin intervención de catadores.</p></div>")
    with gr.Row(equal_height=False):
        with gr.Column(scale=1, elem_classes="panel"):
            gr.HTML("<div class='titulo'>Datos del vino</div>")
            pais = gr.Dropdown(L["paises"], label="País", value=None)
            prov = gr.Dropdown([], label="Provincia / estado", value=None)
            reg = gr.Dropdown([], label="Región", value=None)
            bod = gr.Dropdown([], label="Bodega", value=None)
            var = gr.Dropdown([], label="Variedad", value=None)
            des = gr.Dropdown([], label="Designación", value=None)
            with gr.Row():
                precio = gr.Dropdown(OP_PRECIO, label="Precio", value=None)
                anio = gr.Dropdown(OP_ANIO, label="Añada", value=None)
            btn = gr.Button("Calcular estimación", elem_id="btn")
        with gr.Column(scale=1, elem_classes="panel"):
            gr.HTML("<div class='titulo'>Resultado</div>")
            salida = gr.HTML("<div class='vacio'>Complete los campos y presione «Calcular estimación».</div>")
    gr.HTML(PIE)
    pais.change(on_pais, pais, [prov, reg, bod, var, des])
    prov.change(on_prov, prov, [reg, bod, des])
    reg.change(on_reg, [prov, reg], [bod, des])
    bod.change(on_bod, bod, des)
    btn.click(predecir, [pais, prov, reg, bod, var, des, precio, anio], salida)

demo.launch(share=True)

/tmp/ipykernel_2815/2816074691.py:102: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=CSS, title="Estimador de Calidad de Vinos", theme=gr.themes.Base()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2d4fff1a2e428d8951.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


presentacion web

In [ ]:
# @title

%%html
<!DOCTYPE html>
<html lang="es"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Del dataset limpio al modelo predictivo - Calidad de Vinos</title>
<style>
:root{--bg:#f4f1ec;--c:#fff;--tx:#1f2430;--mu:#6a7080;--ln:#d9d4cb;--ac:#5c001e;--g0:#a0722a;--g1:#2f5d7c;--g2:#2f6b4a}
*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--tx);font:15px/1.6 "Helvetica Neue",Arial,sans-serif}
.w{max-width:980px;margin:0 auto;padding:0 20px}
header{text-align:center;padding:40px 0 24px;border-bottom:1px solid var(--ln)}
.sup{font-size:12px;letter-spacing:.18em;color:var(--ac);font-weight:600}
h1{font:600 34px Georgia,serif;margin:8px 0; color: var(--tx) !important; opacity: 1 !important;}header p{color:var(--mu);max-width:680px;margin:0 auto}
.flow{display:flex;gap:0;margin:26px 0 8px;flex-wrap:wrap}.st{flex:1;min-width:150px;background:var(--c);border:1px solid var(--ln);padding:14px;text-align:center;font-size:13px;margin:-1px 0 0 -1px}
.st b{display:block;color:var(--ac);font-size:12px;letter-spacing:.1em}
section{margin:34px 0}h2{font:600 22px Georgia,serif;margin:0 0 4px; color: var(--tx) !important; opacity: 1 !important;}h2 span{color:var(--ac);font-size:13px;letter-spacing:.12em;display:block;font-family:Arial}
.lead{color:var(--mu);margin:4px 0 14px}
.card{background:var(--c);border:1px solid var(--ln);border-radius:6px;padding:20px;margin-bottom:14px}
table{width:100%;border-collapse:collapse;font-size:13.5px}th,td{padding:9px 10px;border-bottom:1px solid #e7e3dc;text-align:left;vertical-align:top}
th{font-size:11.5px;letter-spacing:.08em;text-transform:uppercase;color:var(--mu)}tr.best td{background:#fbe9ec;font-weight:600}
code{background:#efebe4;padding:1px 6px;border-radius:3px;font-size:12.5px}pre{background:#efebe4;padding:12px;border-radius:4px;overflow-x:auto;font-size:12.5px;margin:10px 0 0}
.g2{display:grid;grid-template-columns:1fr 1fr;gap:14px}@media(max-width:760px){.g2{grid-template-columns:1fr}}
ul{margin:6px 0 0;padding-left:20px}li{margin:4px 0}
.bar{display:flex;align-items:center;gap:10px;margin:8px 0;font-size:13px}.bar span{width:150px}.bar div{flex:1;height:14px;background:#e7e3dc;border-radius:2px;overflow:hidden}.bar i{display:block;height:100%;background:var(--ac)}.bar b{width:52px;text-align:right}
#sc{display:flex;align-items:flex-end;gap:5px;height:190px;margin-top:10px}.col{flex:1;display:flex;flex-direction:column;align-items:center;justify-content:flex-end;height:100%;font-size:10.5px;color:var(--mu)}
.col i{display:block;width:100%;background:#b9b4aa;border-radius:2px 2px 0 0}.col.k i{background:var(--ac)}.col em{font-style:normal;margin-top:4px;color:var(--tx)}
.lib{display:grid;grid-template-columns:repeat(auto-fill,minmax(280px,1fr));gap:12px}.lib div{border:1px solid var(--ln);border-left:3px solid #c22d41;padding:10px 12px;background:var(--c);font-size:13.5px}
.lib b{display:block}.lib small{color:var(--mu)}
.hl{border:2px solid var(--ac)}
footer{text-align:center;border-top:1px solid var(--ln);margin-top:36px;padding:22px 0 34px;font-size:13px;color:var(--mu)}
.eq{display:flex;justify-content:center;gap:28px;flex-wrap:wrap;margin-top:10px;color:var(--tx)}.ln{font-weight:700}
</style></head><body>
<header><div class="w"><div class="sup">TRABAJO PRÁCTICO - CIENCIA DE DATOS</div><h1>Del dataset limpio al modelo predictivo</h1>
<p>El equipo realizó la limpieza base del dataset Wine Reviews en OpenRefine para el análisis exploratorio. Sobre esa base se adaptó el dataset con OpenRefine y pandas y se construyó un modelo que estima la puntuación de un vino y lo clasifica como primer filtro de calidad.</p>
<div class="flow"><div class="st"><b>1. LIMPIEZA BASE</b>OpenRefine, análisis exploratorio</div><div class="st"><b>2. ADAPTACIÓN</b>OpenRefine para el modelo</div><div class="st"><b>3. PANDAS</b>Variables del modelo</div><div class="st"><b>4. MODELOS</b>Entrenamiento y resultados</div><div class="st"><b>5. EXPORTACIÓN</b>Archivos guardados</div><div class="st"><b>6. WEB</b>Predictor con ayuda de IA</div></div></div></header>
<main class="w">

<section><h2><span>PASO 1</span>Limpieza base del equipo (análisis exploratorio)</h2>
<p class="lead">Punto de partida: 129.970 reseñas. Según el reporte técnico del equipo (Grupo trubo), esta etapa se hizo en OpenRefine 3.10 con GREL, priorizando la imputación segura de nulos, la reducción de columnas redundantes y la estandarización de entidades, y es la que alimenta el análisis exploratorio.</p>
<div class="card"><table>
<tr><th>Tarea</th><th>Qué se hizo</th></tr>
<tr><td>Registros vacíos</td><td>Se eliminó el único registro sin <code>variety</code> (facet por vacíos), por no tener peso estadístico.</td></tr>
<tr><td>Columna redundante</td><td>Se eliminó <code>taster_twitter_handle</code>, redundante respecto de <code>taster_name</code>.</td></tr>
<tr><td>Bodegas</td><td>Clustering de <code>winery</code> con Key Collision (fingerprint) y Nearest Neighbor (Levenshtein, radio 2,0, bloque de 6 caracteres) para unificar distintas escrituras de una misma bodega.</td></tr>
<tr><td>Designación</td><td>Unos 30.000 nulos reemplazados por "Sin designación" (vinos de línea estándar), con <code>if(isBlank(value), ...)</code>.</td></tr>
<tr><td>Catador</td><td>Reseñas sin autor agrupadas como "Catador anónimo".</td></tr>
<tr><td>Regiones</td><td>Vacíos de <code>region_1</code> y <code>region_2</code> reemplazados por "Sin región" y "Sin subregión".</td></tr>
<tr><td>Vinos sin añada</td><td>Se creó <code>is_non_vintage</code> (0/1). Los nulos de <code>vintage_year</code> son vinos non-vintage reales, por lo que se dejaron vacíos para conservar la columna numérica y se capturó el efecto con la variable indicadora.</td></tr>
<tr><td>Etapa anterior</td><td>Generación de las columnas que usa el análisis exploratorio: <code>vintage_year</code>, <code>price_tier</code> y el perfil sensorial.</td></tr></table></div></section>

<section><h2><span>PASO 2</span>Adaptación del dataset para el modelo (OpenRefine)</h2>
<p class="lead">Los modelos necesitan variables numéricas, por lo que se tomó el dataset limpio del equipo y se lo transformó, otra vez con OpenRefine.</p>
<div class="card"><table>
<tr><th>Transformación</th><th>Detalle</th></tr>
<tr><td>Perfil sensorial a columnas</td><td>La columna de texto se convirtió en 11 columnas binarias (0/1) con GREL: Seco, Roble / Madera, Mora, Fresco, Especiado, Equilibrado / Sutil, Dulce, Ciruela, Cereza, Buena acidez y Taninos.</td></tr>
<tr><td>País a binarias</td><td>Los vacíos de <code>country</code> se marcaron como "Null" y el país se codificó en 44 columnas (<code>country_binary 1</code> a <code>44</code>) mediante una expresión GREL y la división de columna.</td></tr>
<tr><td>Precio</td><td><code>price_tier_num</code> (1 a 4) como versión numérica del segmento de precio.</td></tr>
<tr><td>Columnas descartadas</td><td>Se eliminaron las de texto libre o sin valor predictivo: <code>description</code>, <code>title</code>, <code>taster_name</code>, la traducción, <code>perfil_sensorial</code>, <code>price_tier</code> y <code>country</code>.</td></tr></table>
<p class="lead" style="margin:10px 0 0">Resultado: un CSV de 129.970 filas y 66 columnas listo para pandas.</p></div></section>

<section><h2><span>PASO 3</span>Complemento con pandas</h2>
<p class="lead">OpenRefine dejó el dataset estructurado; pandas armó las variables finales del modelo.</p>
<div class="g2"><div class="card"><b>Preparación básica</b><ul><li>Reconstrucción del país a partir de las 44 columnas binarias.</li><li>Precio faltante marcado con <code>precio_faltante</code> y reemplazado por la mediana (en escala logarítmica).</li><li>Año de cosecha limitado entre 1990 y 2017.</li><li>División 80 % entrenamiento / 20 % prueba (<code>random_state=42</code>).</li></ul></div>
<div class="card"><b>Variables agregadas</b><ul><li>Frecuencia de bodega, variedad, provincia y región.</li><li>Precio relativo a su variedad, provincia y país.</li><li>Precio medio de la bodega y variedades que produce.</li><li>Palabras clave de la designación (Reserva, Grand Cru, Vieilles Vignes, Estate, Brut).</li></ul></div></div>
<div class="card"><b>Decisión importante: quitamos el perfil sensorial.</b> Esas columnas salen de la reseña de un catador, por lo que no existen antes de catar el vino y mezclan parte de la nota con la variable. Al sacarlas, el acierto bajó apenas 1 o 2 puntos, así que el modelo final usa solo datos previos a la cata: origen, bodega, variedad, designación, precio y añada.</div></section>

<section><h2><span>PASO 4</span>Modelos y resultados</h2>
<p class="lead">El modelo predice el número (puntaje) y a partir de él se asigna el grupo: Regular (86 o menos), Bueno (87 a 90) y Excelente (91 o más).</p>
<div class="card"><table><tr><th>Modelo</th><th>Grupo acertado</th><th>Falsos excelentes</th><th>Joyas ocultas</th></tr>
<tr><td>Ridge (lineal)</td><td>62,8 %</td><td>92</td><td>39</td></tr>
<tr><td>Random Forest</td><td>66,4 %</td><td>75</td><td>42</td></tr>
<tr><td>Extra Trees</td><td>66,0 %</td><td>76</td><td>35</td></tr>
<tr><td>HistGradientBoosting</td><td>66,3 %</td><td>69</td><td>46</td></tr>
<tr><td>HGB con variables nuevas</td><td>66,8 %</td><td>59</td><td>45</td></tr>
<tr><td>CatBoost</td><td>66,2 %</td><td>72</td><td>47</td></tr>
<tr class="best"><td>Ensamble HGB + CatBoost (modelo final)</td><td>67,0 %</td><td>64</td><td>39</td></tr></table>
<div id="mb" style="margin-top:14px"></div><div class="sc2" style="font-size:11.5px;color:#6a7080">Las barras parten desde 55 % para destacar las diferencias.</div>
<p class="lead" style="margin:12px 0 0"><b>Falso excelente:</b> vino Regular que el modelo califica como Excelente (riesgo de compra). <b>Joya oculta:</b> vino Excelente que el modelo califica como Regular (oportunidad perdida). Con el perfil sensorial, el mejor modelo llegó a 58 % de acierto dentro de ±1 punto y 81 % dentro de ±2.</p></div>
<div class="card"><b>¿Por qué un 67 % y no más?</b> La nota de un vino depende en buena parte del catador y de su redacción, información que un filtro previo a la cata no tiene. Probamos más variables, otros algoritmos, clasificación directa y ajuste de umbrales, y todos quedaron entre 66 % y 67,5 %.</div></section>

<section><h2><span>DIFICULTAD POR PUNTAJE</span>Dónde le cuesta clasificar al modelo</h2>
<p class="lead">Porcentaje de vinos bien clasificados según su puntaje real (modelo HGB, conjunto de prueba). Las barras oscuras marcan los puntajes pegados a un límite entre grupos.</p>
<div class="card"><div id="sc"></div>
<ul><li>Los puntajes 86 y 91, justo en el borde de un grupo, son los más difíciles: se clasifican bien solo el 43 % y el 36 % de las veces.</li>
<li>El 99 % de los errores cae en el grupo vecino. El modelo casi no confunde un Regular con un Excelente, y el error típico es de unos 2 puntos.</li>
<li>El grupo Bueno (47 % de los vinos) es el más fácil, con 77 % de aciertos. Regular y Excelente rondan el 58 %.</li>
<li>Los puntajes extremos (menos de 82 o más de 94) se aciertan mejor porque están lejos de los límites.</li></ul></div></section>

<section><h2><span>PASO 5</span>Exportación de los modelos</h2>
<div class="card"><p style="margin-top:0">Reentrenamos con el 100 % de los datos y guardamos todo en Google Drive para usarlo sin volver a entrenar:</p>
<table><tr><td><code>hgb.joblib</code></td><td>Modelo HistGradientBoosting con su preprocesamiento.</td></tr><tr><td><code>catboost.cbm</code></td><td>Modelo CatBoost.</td></tr><tr><td><code>referencias.joblib</code></td><td>Tablas auxiliares, error típico y listas de opciones para la web.</td></tr><tr><td><code>utilidades.py</code></td><td>Función que arma las variables, idéntica en entrenamiento y en la web.</td></tr><tr><td><code>resultados.csv</code></td><td>Tabla de métricas.</td></tr></table></div></section>

<section><h2><span>PASO 6</span>Web con ayuda de inteligencia artificial</h2>
<div class="card">La página predictora (desplegables en cascada: país, provincia, región, bodega, variedad, designación, precio y añada) se programó con la asistencia de <b>Claude (Anthropic)</b>, a partir de nuestras especificaciones y de los modelos exportados. Devuelve la puntuación estimada, su rango probable, el grupo y la probabilidad de cada grupo. La IA se usó como herramienta de programación; la limpieza, el análisis y la elección de los modelos fueron decisiones del equipo.</div></section>

<section><h2><span>HERRAMIENTAS</span>Librerías y software utilizados</h2>
<div class="lib">
<div><b>OpenRefine 3.10</b><small>Limpieza base (equipo) y adaptación para el modelo: GREL, clustering, columnas binarias.</small></div>
<div><b>pandas</b><small>Carga del CSV, variables y agrupaciones.</small></div>
<div><b>NumPy</b><small>Cálculo numérico, logaritmo del precio, métricas.</small></div>
<div><b>scikit-learn</b><small>Ridge, RandomForest, ExtraTrees, HistGradientBoosting, TargetEncoder, ColumnTransformer, Pipeline, train_test_split, confusion_matrix.</small></div>
<div><b>CatBoost</b><small>Modelo de boosting con variables categóricas nativas.</small></div>
<div><b>joblib</b><small>Guardado y carga de los modelos.</small></div>
<div><b>Plotly Express</b><small>Gráficos del análisis exploratorio realizado por el equipo (boxplot, barras, dispersión, sunburst).</small></div>
<div><b>Gradio</b><small>Interfaz web del predictor.</small></div>
<div><b>Google Colab y Google Drive</b><small>Entorno de ejecución y almacenamiento.</small></div>
<div><b>Biblioteca estándar de Python</b><small>time, os, json, math, html, importlib, subprocess.</small></div>
<div><b>API MyMemory</b><small>Prueba de traducción (descartada).</small></div>
<div><b>Claude (Anthropic)</b><small>Asistente de IA para código y página web.</small></div></div></section>
</main>
<footer><div class="w">Dataset: Wine Reviews (≈130.000 reseñas). Documento elaborado con la asistencia de Claude (Anthropic).
<div class="eq"><div class="member"><span class="ln">CAMPOS</span>, Fiorella</div><div class="member"><span class="ln">MARECO</span>, Carlos</div><div class="member"><span class="ln">SALVATORE</span>, Santiago</div><div class="member"><span class="ln">SITTNER</span>, Germán</div></div></div></footer>
<script>
const m=[["Ridge",62.8],["Random Forest",66.4],["Extra Trees",66.0],["HistGradientBoosting",66.3],["HGB + variables nuevas",66.8],["CatBoost",66.2],["Ensamble final",67.0]];
document.getElementById('mb').innerHTML=m.map(([n,v])=>`<div class="bar"><span>${n}</span><div><i style="width:${(v-55)/15*100}%"></i></div><b>${v}%</b></div>`).join('');
const s={80:84.2,81:87.4,82:74.6,83:77.1,84:67.5,85:55.5,86:42.7,87:70.2,88:80.5,89:82.5,90:76.1,91:36.2,92:54.4,93:72.0,94:85.6,95:91.8,96:99.1};
document.getElementById('sc').innerHTML=Object.entries(s).map(([k,v])=>`<div class="col${[86,87,90,91].includes(+k)?' k':''}">${Math.round(v)}<i style="height:${v*1.4}px"></i><em>${k}</em></div>`).join('');
</script></body></html>

Tarea,Qué se hizo
Registros vacíos,"Se eliminó el único registro sin variety (facet por vacíos), por no tener peso estadístico."
Columna redundante,"Se eliminó taster_twitter_handle, redundante respecto de taster_name."
Bodegas,"Clustering de winery con Key Collision (fingerprint) y Nearest Neighbor (Levenshtein, radio 2,0, bloque de 6 caracteres) para unificar distintas escrituras de una misma bodega."
Designación,"Unos 30.000 nulos reemplazados por ""Sin designación"" (vinos de línea estándar), con if(isBlank(value), ...)."
Catador,"Reseñas sin autor agrupadas como ""Catador anónimo""."
Regiones,"Vacíos de region_1 y region_2 reemplazados por ""Sin región"" y ""Sin subregión""."
Vinos sin añada,"Se creó is_non_vintage (0/1). Los nulos de vintage_year son vinos non-vintage reales, por lo que se dejaron vacíos para conservar la columna numérica y se capturó el efecto con la variable indicadora."
Etapa anterior,"Generación de las columnas que usa el análisis exploratorio: vintage_year, price_tier y el perfil sensorial."
Transformación,Detalle
Perfil sensorial a columnas,"La columna de texto se convirtió en 11 columnas binarias (0/1) con GREL: Seco, Roble / Madera, Mora, Fresco, Especiado, Equilibrado / Sutil, Dulce, Ciruela, Cereza, Buena acidez y Taninos."
